# IF3170 Artificial Intelligence | Tugas Besar 2

This notebook serves as a template for the assignment. Please create a copy of this notebook to complete your work. You can add more code blocks, markdown blocks, or new sections if needed.


Group Number: 06

Group Members:
- Richard Christian 13523024
- Kenneth Poenadi Name 13523040
- Ivan Wirawan 13523046
- Bob Kunanda 13523086
- M Zahran Ramadhan 13523104

## Import Libraries

In [155]:
%pip install pydantic ydata-profiling

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import pickle
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 67
np.random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Import untuk cross-validation
from sklearn.model_selection import StratifiedKFold

print("[OK] Libraries imported successfully")
print(f"Random state: {RANDOM_STATE}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

Note: you may need to restart the kernel to use updated packages.
[OK] Libraries imported successfully
Random state: 67
NumPy version: 1.25.2
Pandas version: 2.2.3


## Import Dataset

In [156]:
df = pd.read_csv("../data/train.csv")

print("="*100)
print("DATASET OVERVIEW")
print("="*100)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes.value_counts()}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nTarget distribution:")
print(df['Target'].value_counts())
print("="*100)

DATASET OVERVIEW
Shape: 3096 rows x 38 columns

Columns: ['Student_ID', 'Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance\t', 'Previous qualification', 'Previous qualification (grade)', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Admission grade', 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations

# Exploratory Data Analysis (Optional)

Exploratory Data Analysis (EDA) is a crucial step in the data analysis process that involves examining and visualizing data sets to uncover patterns, trends, anomalies, and insights. It is the first step before applying more advanced statistical and machine learning techniques. EDA helps you to gain a deep understanding of the data you are working with, allowing you to make informed decisions and formulate hypotheses for further analysis.

In [157]:
# profile = ProfileReport(df, title="Data Profiling Report", explorative=True)
# profile.to_file("../data/profile_report.html")

# 1. Split Training Set and Validation Set

Splitting the training and validation set works as an early diagnostic towards the performance of the model we train. This is done before the preprocessing steps to **avoid data leakage inbetween the sets**. If you want to use k-fold cross-validation, split the data later and do the cleaning and preprocessing separately for each split.

Note: For training, you should use the data contained in the `train` folder given by the TA. The `test` data is only used for kaggle submission.

In [158]:
print("="*100)
print("SPLIT TRAINING AND VALIDATION SET")
print("="*100)

# Separate features and target (remove Student_ID as it's not a feature)
X = df.drop(columns=['Target', 'Student_ID'])
y = df['Target']

print(f"Original dataset: {len(df)} samples")
print(f"Target distribution:")
print(y.value_counts())
print(f"\nTarget proportions:")
print(y.value_counts(normalize=True))

# Split dengan stratified sampling (80% train, 20% validation)
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=RANDOM_STATE, 
    stratify=y  # Maintain class distribution
)

print(f"\n[Stats] After split:")
print(f"Training set: {len(X_train)} samples ({len(X_train)/len(df)*100:.1f}%)")
print(f"Validation set: {len(X_val)} samples ({len(X_val)/len(df)*100:.1f}%)")

print(f"\n[Stats] Training set target distribution:")
print(y_train.value_counts())
print(f"\n[Stats] Validation set target distribution:")
print(y_val.value_counts())

print("\n[OK] Data split completed with stratified sampling")
print("="*100)

SPLIT TRAINING AND VALIDATION SET
Original dataset: 3096 samples
Target distribution:
Target
Graduate    1546
Dropout      994
Enrolled     556
Name: count, dtype: int64

Target proportions:
Target
Graduate    0.499354
Dropout     0.321059
Enrolled    0.179587
Name: proportion, dtype: float64

[Stats] After split:
Training set: 2476 samples (80.0%)
Validation set: 620 samples (20.0%)

[Stats] Training set target distribution:
Target
Graduate    1236
Dropout      795
Enrolled     445
Name: count, dtype: int64

[Stats] Validation set target distribution:
Target
Graduate    310
Dropout     199
Enrolled    111
Name: count, dtype: int64

[OK] Data split completed with stratified sampling


In [159]:
# Check actual column names
print("Actual columns in dataset:")
for i, col in enumerate(X.columns, 1):
    print(f"{i}. {col}")

Actual columns in dataset:
1. Marital status
2. Application mode
3. Application order
4. Course
5. Daytime/evening attendance	
6. Previous qualification
7. Previous qualification (grade)
8. Nacionality
9. Mother's qualification
10. Father's qualification
11. Mother's occupation
12. Father's occupation
13. Admission grade
14. Displaced
15. Educational special needs
16. Debtor
17. Tuition fees up to date
18. Gender
19. Scholarship holder
20. Age at enrollment
21. International
22. Curricular units 1st sem (credited)
23. Curricular units 1st sem (enrolled)
24. Curricular units 1st sem (evaluations)
25. Curricular units 1st sem (approved)
26. Curricular units 1st sem (grade)
27. Curricular units 1st sem (without evaluations)
28. Curricular units 2nd sem (credited)
29. Curricular units 2nd sem (enrolled)
30. Curricular units 2nd sem (evaluations)
31. Curricular units 2nd sem (approved)
32. Curricular units 2nd sem (grade)
33. Curricular units 2nd sem (without evaluations)
34. Unemployment r

# 2. Data Cleaning and Preprocessing

This step is the first thing to be done once a Data Scientist have grasped a general knowledge of the data. Raw data is **seldom ready for training**, therefore steps need to be taken to clean and format the data for the Machine Learning model to interpret.

By performing data cleaning and preprocessing, you ensure that your dataset is ready for model training, leading to more accurate and reliable machine learning results. These steps are essential for transforming raw data into a format that machine learning algorithms can effectively learn from and make predictions.

We will give some common methods for you to try, but you only have to **at least implement one method for each process**. For each step that you will do, **please explain the reason why did you do that process. Write it in a markdown cell under the code cell you wrote.**

## A. Data Cleaning

**Data cleaning** is the crucial first step in preparing your dataset for machine learning. Raw data collected from various sources is often messy and may contain errors, missing values, and inconsistencies. Data cleaning involves the following steps:

1. **Handling Missing Data:** Identify and address missing values in the dataset. This can include imputing missing values, removing rows or columns with excessive missing data, or using more advanced techniques like interpolation.

2. **Dealing with Outliers:** Identify and handle outliers, which are data points significantly different from the rest of the dataset. Outliers can be removed or transformed to improve model performance.

3. **Data Validation:** Check for data integrity and consistency. Ensure that data types are correct, categorical variables have consistent labels, and numerical values fall within expected ranges.

4. **Removing Duplicates:** Identify and remove duplicate rows, as they can skew the model's training process and evaluation metrics.

5. **Feature Engineering**: Create new features or modify existing ones to extract relevant information. This step can involve scaling, normalizing, or encoding features for better model interpretability.

### I. Handling Missing Data

Missing data can adversely affect the performance and accuracy of machine learning models. There are several strategies to handle missing data in machine learning:

1. **Data Imputation:**

    a. **Mean, Median, or Mode Imputation:** For numerical features, you can replace missing values with the mean, median, or mode of the non-missing values in the same feature. This method is simple and often effective when data is missing at random.

    b. **Constant Value Imputation:** You can replace missing values with a predefined constant value (e.g., 0) if it makes sense for your dataset and problem.

    c. **Imputation Using Predictive Models:** More advanced techniques involve using predictive models to estimate missing values. For example, you can train a regression model to predict missing numerical values or a classification model to predict missing categorical values.

2. **Deletion of Missing Data:**

    a. **Listwise Deletion:** In cases where the amount of missing data is relatively small, you can simply remove rows with missing values from your dataset. However, this approach can lead to a loss of valuable information.

    b. **Column (Feature) Deletion:** If a feature has a large number of missing values and is not critical for your analysis, you can consider removing that feature altogether.

3. **Domain-Specific Strategies:**

    a. **Domain Knowledge:** In some cases, domain knowledge can guide the imputation process. For example, if you know that missing values are related to a specific condition, you can impute them accordingly.

4. **Imputation Libraries:**

    a. **Scikit-Learn:** Scikit-Learn provides a `SimpleImputer` class that can handle basic imputation strategies like mean, median, and mode imputation.

    b. **Fancyimpute:** Fancyimpute is a Python library that offers more advanced imputation techniques, including matrix factorization, k-nearest neighbors, and deep learning-based methods.

The choice of imputation method should be guided by the nature of your data, the amount of missing data, the problem you are trying to solve, and the assumptions you are willing to make.

In [160]:
categorical_cols = [
    'Marital status', 'Application mode', 'Application order', 'Course',
    'Previous qualification', 'Nationality', 
    "Mother's qualification", "Father's qualification", 
    "Mother's occupation", "Father's occupation", 'Educational special needs', 
    'International', 'Debtor', 'Tuition fees up to date', 'Scholarship holder', 
    'Displaced', 'Gender'
]

# Check if 'Daytime/evening attendance' exists in the data
if 'Daytime/evening attendance' in X_train.columns:
    categorical_cols.append('Daytime/evening attendance')
elif 'Daytime/evening attendance\t' in X_train.columns:
    categorical_cols.append('Daytime/evening attendance\t')

continuous_cols = [
    'Previous qualification (grade)', 'Admission grade', 'Age at enrollment',
    'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)',
    'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)',
    'Unemployment rate', 'Inflation rate', 'GDP'
]

print("="*100)
print("HANDLING MISSING DATA")
print("="*100)

print(f"Total features: {len(X_train.columns)}")
print(f"Categorical features: {len(categorical_cols)}")
print(f"Continuous features: {len(continuous_cols)}")
print()

# Check missing values in training set
missing_train = X_train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)

print(f"Missing values in training set:")
if len(missing_train) > 0:
    print(missing_train)
    print(f"\nTotal columns with missing values: {len(missing_train)}")
else:
    print("[OK] No missing values found")

# Check missing values in validation set
missing_val = X_val.isnull().sum()
missing_val = missing_val[missing_val > 0].sort_values(ascending=False)

print(f"\nMissing values in validation set:")
if len(missing_val) > 0:
    print(missing_val)
else:
    print("[OK] No missing values found")

print("\n" + "="*100)
print("CONCLUSION: Dataset is clean with no missing values")
print("="*100)

HANDLING MISSING DATA
Total features: 36
Categorical features: 18
Continuous features: 18

Missing values in training set:
[OK] No missing values found

Missing values in validation set:
[OK] No missing values found

CONCLUSION: Dataset is clean with no missing values


### II. Dealing with Outliers

Outliers are data points that significantly differ from the majority of the data. They can be unusually high or low values that do not fit the pattern of the rest of the dataset. Outliers can significantly impact model performance, so it is important to handle them properly.

Some methods to handle outliers:
1. **Imputation**: Replace with mean, median, or a boundary value.
2. **Clipping**: Cap values to upper and lower limits.
3. **Transformation**: Use log, square root, or power transformations to reduce their influence.
4. **Model-Based**: Use algorithms robust to outliers (e.g., tree-based models, Huber regression).

**[Note] Dalam implementasi ini:**
- Menggunakan **IQR (Interquartile Range) Method** untuk deteksi outliers
- Penanganan dengan **CLIPPING** (bukan removal) agar tidak kehilangan data
- Hanya berlaku untuk **kolom NUMERICAL**, tidak untuk categorical
- Cell berikutnya akan menampilkan detail outlier per kolom

In [161]:
print("="*100)
print("OUTLIER DETECTION AND HANDLING - NUMERICAL FEATURES")
print("="*100)

X_train_clean = X_train.copy()
X_val_clean = X_val.copy()

outlier_info = {}
numerical_cols = continuous_cols.copy()
numerical_cols_to_check = continuous_cols.copy()

print("Method: IQR (Interquartile Range)")
print("Handling: CLIPPING (cap to lower/upper bounds)")
print("\nAdaptive factors - MORE LENIENT to preserve dropout signals:")
print("  * Age: factor=6.0 (very loose - age variation normal)")
print("  * Curricular columns: factor=5.0, lower_bound=0.0 (zeros valid)")
print("  * Credited/without evaluation: SKIP (mostly zeros)")
print("  * Default: factor=3.5 (looser than before)")
print("-"*100)

for col in continuous_cols:
    if col not in X_train_clean.columns:
        continue
    
    # Skip credited and without evaluation columns
    if 'credited' in col.lower() or 'without evaluation' in col.lower():
        print(f"[SKIP] {col}: SKIPPED (mostly zeros)")
        continue
    
    # Calculate IQR
    Q1 = X_train_clean[col].quantile(0.25)
    Q3 = X_train_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # Adaptive factor - MORE LENIENT
    if 'age' in col.lower():
        factor = 6.0  # Was 4.0
    elif 'curricular' in col.lower():
        factor = 5.0  # Was 4.0
    else:
        factor = 3.5  # Was 2.5
    
    lower_bound = Q1 - factor * IQR
    upper_bound = Q3 + factor * IQR
    
    # For curricular columns, ensure lower bound is 0
    if 'curricular' in col.lower():
        lower_bound = 0.0
    
    # Store bounds
    outlier_info[col] = {
        'lower_bound': lower_bound,
        'upper_bound': upper_bound,
        'factor': factor,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR
    }
    
    # Count and clip outliers in training
    outliers_train = ((X_train_clean[col] < lower_bound) | (X_train_clean[col] > upper_bound)).sum()
    X_train_clean[col] = X_train_clean[col].clip(lower_bound, upper_bound)
    
    # Count and clip outliers in validation
    outliers_val = ((X_val_clean[col] < lower_bound) | (X_val_clean[col] > upper_bound)).sum()
    X_val_clean[col] = X_val_clean[col].clip(lower_bound, upper_bound)
    
    if outliers_train > 0 or outliers_val > 0:
        print(f"[OK] {col}")
        print(f"  Bounds: [{lower_bound:.2f}, {upper_bound:.2f}] (factor={factor})")
        print(f"  Train: {outliers_train} outliers clipped")
        print(f"  Val: {outliers_val} outliers clipped")

print("\n" + "="*100)
print(f"[OK] OUTLIER HANDLING COMPLETED")
print(f"  Columns processed: {len([c for c in continuous_cols if c in outlier_info])}")
print(f"  Columns skipped: {len([c for c in continuous_cols if 'credited' in c.lower() or 'without evaluation' in c.lower()])}")
print("="*100)

OUTLIER DETECTION AND HANDLING - NUMERICAL FEATURES
Method: IQR (Interquartile Range)
Handling: CLIPPING (cap to lower/upper bounds)

Adaptive factors - MORE LENIENT to preserve dropout signals:
  * Age: factor=6.0 (very loose - age variation normal)
  * Curricular columns: factor=5.0, lower_bound=0.0 (zeros valid)
  * Credited/without evaluation: SKIP (mostly zeros)
  * Default: factor=3.5 (looser than before)
----------------------------------------------------------------------------------------------------
[OK] Age at enrollment
  Bounds: [-17.00, 61.00] (factor=6.0)
  Train: 1 outliers clipped
  Val: 1 outliers clipped
[SKIP] Curricular units 1st sem (credited): SKIPPED (mostly zeros)
[OK] Curricular units 1st sem (enrolled)
  Bounds: [0.00, 17.00] (factor=5.0)
  Train: 19 outliers clipped
  Val: 3 outliers clipped
[OK] Curricular units 1st sem (evaluations)
  Bounds: [0.00, 30.00] (factor=5.0)
  Train: 4 outliers clipped
  Val: 0 outliers clipped
[OK] Curricular units 1st sem (ap

**Penjelasan Outlier Handling:**

Menggunakan **IQR Method dengan Adaptive Factors - LENIENT STRATEGY**:
- **Age**: Factor 6.0 (sangat loose - range 18-70 tahun wajar untuk student)
- **Curricular columns**: Factor 5.0 dengan lower bound 0.0 (nilai 0 valid, failing grade bukan outlier)
- **Credited/Without evaluation**: Di-skip karena kebanyakan bernilai 0 
- **Default**: Factor 3.5 (lebih lenient - extreme values bisa jadi signal penting untuk dropout)

**Menggunakan CLIPPING** bukan removal karena:
1. Tidak kehilangan data (important untuk dataset kecil)
2. Tetap menjaga informasi bahwa ada "extreme value"
3. Extreme academic performance (very high/low) adalah dropout predictor yang kuat
3. Lebih robust untuk machine learning

### III. Remove Duplicates

Handling duplicate values is crucial because they can compromise data integrity, leading to inaccurate analysis and insights. Duplicate entries can bias machine learning models, causing overfitting and reducing their ability to generalize to new data. They also inflate the dataset size unnecessarily, increasing computational costs and processing times. Additionally, duplicates can distort statistical measures and lead to inconsistencies, ultimately affecting the reliability of data-driven decisions and reporting. Ensuring data quality by removing duplicates is essential for accurate, efficient, and consistent analysis.

**[Note] Cell berikutnya akan:**
- Menghitung jumlah duplicate rows
- Menjelaskan apa itu duplicates dan kenapa harus dihapus
- Menghapus duplicate rows (jika ada)
- Menampilkan statistik sebelum vs sesudah

In [162]:
print("="*100)
print("REMOVE DUPLICATES")
print("="*100)

print("What are duplicates?")
print("-> Rows yang memiliki nilai IDENTIK di SEMUA kolom")
print("\nWhy remove duplicates?")
print("-> Duplikat dapat menyebabkan overfitting (model 'menghapal' data yang sama)")
print("-> Mengurangi ukuran dataset tanpa kehilangan informasi unik")
print("-> Meningkatkan efisiensi komputasi")
print("-"*100)

# Check duplicates in training set
duplicates_train_before = X_train_clean.duplicated().sum()
total_train_before = len(X_train_clean)

print(f"\n[Stats] Training set BEFORE:")
print(f"  Total rows: {total_train_before}")
print(f"  Duplicate rows: {duplicates_train_before}")
print(f"  Percentage: {duplicates_train_before/total_train_before*100:.2f}%")

if duplicates_train_before > 0:
    # Remove duplicates
    X_train_clean = X_train_clean.drop_duplicates()
    y_train = y_train[X_train_clean.index]
    
    total_train_after = len(X_train_clean)
    print(f"\n[Stats] Training set AFTER:")
    print(f"  Total rows: {total_train_after}")
    print(f"  Rows removed: {duplicates_train_before}")
    print(f"  [OK] Duplicates removed")
else:
    print("\n[OK] No duplicates found in training set")

# Check duplicates in validation set
duplicates_val_before = X_val_clean.duplicated().sum()
total_val_before = len(X_val_clean)

print(f"\n[Stats] Validation set BEFORE:")
print(f"  Total rows: {total_val_before}")
print(f"  Duplicate rows: {duplicates_val_before}")
print(f"  Percentage: {duplicates_val_before/total_val_before*100:.2f}%")

if duplicates_val_before > 0:
    X_val_clean = X_val_clean.drop_duplicates()
    y_val = y_val[X_val_clean.index]
    
    total_val_after = len(X_val_clean)
    print(f"\n[Stats] Validation set AFTER:")
    print(f"  Total rows: {total_val_after}")
    print(f"  Rows removed: {duplicates_val_before}")
    print(f"  [OK] Duplicates removed")
else:
    print("\n[OK] No duplicates found in validation set")

print("\n" + "="*100)
print("[OK] DUPLICATE CHECK COMPLETED")
print("="*100)

REMOVE DUPLICATES
What are duplicates?
-> Rows yang memiliki nilai IDENTIK di SEMUA kolom

Why remove duplicates?
-> Duplikat dapat menyebabkan overfitting (model 'menghapal' data yang sama)
-> Mengurangi ukuran dataset tanpa kehilangan informasi unik
-> Meningkatkan efisiensi komputasi
----------------------------------------------------------------------------------------------------

[Stats] Training set BEFORE:
  Total rows: 2476
  Duplicate rows: 0
  Percentage: 0.00%

[OK] No duplicates found in training set

[Stats] Validation set BEFORE:
  Total rows: 620
  Duplicate rows: 0
  Percentage: 0.00%

[OK] No duplicates found in validation set

[OK] DUPLICATE CHECK COMPLETED


### IV. Feature Engineering

**Feature engineering** involves creating new features (input variables) or transforming existing ones to improve the performance of machine learning models. Feature engineering aims to enhance the model's ability to learn patterns and make accurate predictions from the data. It's often said that "good features make good models."

1. **Feature Selection:** Feature engineering can involve selecting the most relevant and informative features from the dataset. Removing irrelevant or redundant features not only simplifies the model but also reduces the risk of overfitting.

2. **Creating New Features:** Sometimes, the existing features may not capture the underlying patterns effectively. In such cases, engineers create new features that provide additional information. For example:
   
   - **Polynomial Features:** Engineers may create new features by taking the square, cube, or other higher-order terms of existing numerical features. This can help capture nonlinear relationships.
   
   - **Interaction Features:** Interaction features are created by combining two or more existing features. For example, if you have features "length" and "width," you can create an "area" feature by multiplying them.

3. **Binning or Discretization:** Continuous numerical features can be divided into bins or categories. For instance, age values can be grouped into bins like "child," "adult," and "senior."

4. **Domain-Specific Feature Engineering:** Depending on the domain and problem, engineers may create domain-specific features. For example, in fraud detection, features related to transaction history and user behavior may be engineered to identify anomalies.

Feature engineering is both a creative and iterative process. It requires a deep understanding of the data, domain knowledge, and experimentation to determine which features will enhance the model's predictive power.

In [163]:
print("="*100)
print("FEATURE ENGINEERING")
print("="*100)

print("Creating new features from existing data...")
print("Features to create:")
print("  1. Total_Units_Approved - Total SKS yang lulus (sem 1 + sem 2)")
print("  2. Total_Units_Enrolled - Total SKS yang diambil (sem 1 + sem 2)")
print("  3. Approval_Rate - Rasio SKS lulus vs diambil (success rate)")
print("  4. Average_Grade - Rata-rata nilai (sem 1 + sem 2)")
print("  5. Grade_Difference - Perubahan nilai (sem 2 - sem 1)")
print("-"*100)

X_train_fe = X_train_clean.copy()
X_val_fe = X_val_clean.copy()

# 1. Total units approved
if all(col in X_train_fe.columns for col in ['Curricular units 1st sem (approved)', 
                                               'Curricular units 2nd sem (approved)']):
    X_train_fe['Total_Units_Approved'] = (
        X_train_fe['Curricular units 1st sem (approved)'] + 
        X_train_fe['Curricular units 2nd sem (approved)']
    )
    X_val_fe['Total_Units_Approved'] = (
        X_val_fe['Curricular units 1st sem (approved)'] + 
        X_val_fe['Curricular units 2nd sem (approved)']
    )
    numerical_cols.append('Total_Units_Approved')
    print("[OK] Total_Units_Approved created")

# 2. Total units enrolled
if all(col in X_train_fe.columns for col in ['Curricular units 1st sem (enrolled)', 
                                               'Curricular units 2nd sem (enrolled)']):
    X_train_fe['Total_Units_Enrolled'] = (
        X_train_fe['Curricular units 1st sem (enrolled)'] + 
        X_train_fe['Curricular units 2nd sem (enrolled)']
    )
    X_val_fe['Total_Units_Enrolled'] = (
        X_val_fe['Curricular units 1st sem (enrolled)'] + 
        X_val_fe['Curricular units 2nd sem (enrolled)']
    )
    numerical_cols.append('Total_Units_Enrolled')
    print("[OK] Total_Units_Enrolled created")

# 3. Approval rate
if 'Total_Units_Approved' in X_train_fe.columns and 'Total_Units_Enrolled' in X_train_fe.columns:
    X_train_fe['Approval_Rate'] = np.where(
        X_train_fe['Total_Units_Enrolled'] > 0,
        X_train_fe['Total_Units_Approved'] / X_train_fe['Total_Units_Enrolled'],
        0
    )
    X_val_fe['Approval_Rate'] = np.where(
        X_val_fe['Total_Units_Enrolled'] > 0,
        X_val_fe['Total_Units_Approved'] / X_val_fe['Total_Units_Enrolled'],
        0
    )
    numerical_cols.append('Approval_Rate')
    print("[OK] Approval_Rate created")

# 4. Average grade
if all(col in X_train_fe.columns for col in ['Curricular units 1st sem (grade)', 
                                               'Curricular units 2nd sem (grade)']):
    X_train_fe['Average_Grade'] = (
        X_train_fe['Curricular units 1st sem (grade)'] + 
        X_train_fe['Curricular units 2nd sem (grade)']
    ) / 2
    X_val_fe['Average_Grade'] = (
        X_val_fe['Curricular units 1st sem (grade)'] + 
        X_val_fe['Curricular units 2nd sem (grade)']
    ) / 2
    numerical_cols.append('Average_Grade')
    print("[OK] Average_Grade created")

# 5. Grade difference
if all(col in X_train_fe.columns for col in ['Curricular units 1st sem (grade)', 
                                               'Curricular units 2nd sem (grade)']):
    X_train_fe['Grade_Difference'] = (
        X_train_fe['Curricular units 2nd sem (grade)'] - 
        X_train_fe['Curricular units 1st sem (grade)']
    )
    X_val_fe['Grade_Difference'] = (
        X_val_fe['Curricular units 2nd sem (grade)'] - 
        X_val_fe['Curricular units 1st sem (grade)']
    )
    numerical_cols.append('Grade_Difference')
    print("[OK] Grade_Difference created")

print("\n" + "-"*100)
print("SELECTIVE INTERACTION FEATURES (Only high-impact interactions)")
print("-"*100)

# 6. Academic_Performance_Score (comprehensive performance metric)
if all(col in X_train_fe.columns for col in ['Approval_Rate', 'Average_Grade']):
    # Weighted score: approval rate (60%) + normalized grade (40%)
    X_train_fe['Academic_Performance_Score'] = (
        0.6 * X_train_fe['Approval_Rate'] + 
        0.4 * (X_train_fe['Average_Grade'] / X_train_fe['Average_Grade'].max())
    )
    X_val_fe['Academic_Performance_Score'] = (
        0.6 * X_val_fe['Approval_Rate'] + 
        0.4 * (X_val_fe['Average_Grade'] / X_val_fe['Average_Grade'].max())
    )
    numerical_cols.append('Academic_Performance_Score')
    print("[OK] Academic_Performance_Score created (approval 60% + grade 40%)")

# 7. Grade_Trend_Binary (improving vs declining)
if 'Grade_Difference' in X_train_fe.columns:
    X_train_fe['Grade_Trend_Binary'] = (X_train_fe['Grade_Difference'] > 0).astype(int)
    X_val_fe['Grade_Trend_Binary'] = (X_val_fe['Grade_Difference'] > 0).astype(int)
    categorical_cols.append('Grade_Trend_Binary')
    print("[OK] Grade_Trend_Binary created (improving=1 vs declining=0)")

# 8. Academic_Load_Risk (enrollment vs approved interaction)
if all(col in X_train_fe.columns for col in ['Total_Units_Enrolled', 'Total_Units_Approved']):
    X_train_fe['Academic_Load_Risk'] = X_train_fe['Total_Units_Enrolled'] - X_train_fe['Total_Units_Approved']
    X_val_fe['Academic_Load_Risk'] = X_val_fe['Total_Units_Enrolled'] - X_val_fe['Total_Units_Approved']
    numerical_cols.append('Academic_Load_Risk')
    print("[OK] Academic_Load_Risk created (units failed)")

print("\n" + "="*100)
print("[OK] FEATURE ENGINEERING COMPLETED")
print(f"  Basic features created: 5")
print(f"  High-impact interaction features: 3")
print(f"  Total new features: 8")
print(f"  Total numerical features now: {len([c for c in numerical_cols if c in X_train_fe.columns])}")
print(f"  Total categorical features now: {len([c for c in categorical_cols if c in X_train_fe.columns])}")
print("="*100)

# Display sample of new features
print("\n[Stats] Sample of new features (first 5 rows):")
basic_features = ['Total_Units_Approved', 'Total_Units_Enrolled', 'Approval_Rate', 
                  'Average_Grade', 'Grade_Difference']
advanced_features = ['Academic_Performance_Score', 'Academic_Load_Risk']
all_new_features = basic_features + advanced_features
if all(f in X_train_fe.columns for f in all_new_features):
    print(X_train_fe[all_new_features].head())

FEATURE ENGINEERING
Creating new features from existing data...
Features to create:
  1. Total_Units_Approved - Total SKS yang lulus (sem 1 + sem 2)
  2. Total_Units_Enrolled - Total SKS yang diambil (sem 1 + sem 2)
  3. Approval_Rate - Rasio SKS lulus vs diambil (success rate)
  4. Average_Grade - Rata-rata nilai (sem 1 + sem 2)
  5. Grade_Difference - Perubahan nilai (sem 2 - sem 1)
----------------------------------------------------------------------------------------------------
[OK] Total_Units_Approved created
[OK] Total_Units_Enrolled created
[OK] Approval_Rate created
[OK] Average_Grade created
[OK] Grade_Difference created

----------------------------------------------------------------------------------------------------
SELECTIVE INTERACTION FEATURES (Only high-impact interactions)
----------------------------------------------------------------------------------------------------
[OK] Academic_Performance_Score created (approval 60% + grade 40%)
[OK] Grade_Trend_Binary cr

**Penjelasan Feature Engineering:**

Feature engineering adalah proses membuat fitur baru dari fitur yang sudah ada untuk meningkatkan performa model.

**Fitur yang dibuat:**

1. **Total_Units_Approved**: Total SKS yang berhasil lulus di 2 semester
   - Menunjukkan **produktivitas akademik** secara keseluruhan
   
2. **Total_Units_Enrolled**: Total SKS yang diambil di 2 semester
   - Menunjukkan **beban akademik** yang dipilih mahasiswa

3. **Approval_Rate**: Rasio SKS lulus / SKS diambil
   - **Success rate** - indikator paling penting untuk prediksi dropout
   - Value: 0.0 (gagal semua) sampai 1.0 (lulus semua)

4. **Average_Grade**: Rata-rata nilai kedua semester
   - Menunjukkan **kualitas akademik** secara keseluruhan
   
5. **Grade_Difference**: Perubahan nilai dari semester 1 ke 2
   - Positif = **improving** (semakin baik)
   - Negatif = **declining** (semakin buruk)
   - Trend ini penting untuk prediksi dropout

**Mengapa fitur ini berguna?**
- Model dapat menangkap **pattern** yang tidak terlihat dari fitur individual
- Misalnya: mahasiswa dengan grade tinggi tapi declining trend tetap berisiko dropout

## B. Data Preprocessing

**Data preprocessing** is a broader step that encompasses both data cleaning and additional transformations to make the data suitable for machine learning algorithms. Its primary goals are:

1. **Feature Scaling:** Ensure that numerical features have similar scales. Common techniques include Min-Max scaling (scaling to a specific range) or standardization (mean-centered, unit variance).

2. **Encoding Categorical Variables:** Machine learning models typically work with numerical data, so categorical variables need to be encoded. This can be done using one-hot encoding, label encoding, or more advanced methods like target encoding.

3. **Handling Imbalanced Classes:** If dealing with imbalanced classes in a binary classification task, apply techniques such as oversampling, undersampling, or using different evaluation metrics to address class imbalance.

4. **Dimensionality Reduction:** Reduce the number of features using techniques like Principal Component Analysis (PCA) or feature selection to simplify the model and potentially improve its performance.

5. **Normalization:** Normalize data to achieve a standard distribution. This is particularly important for algorithms that assume normally distributed data.

### Notes on Preprocessing processes

It is advised to create functions or classes that have the same/similar type of inputs and outputs, so you can add, remove, or swap the order of the processes easily. You can implement the functions or classes by yourself

or

use `sklearn` library. To create a new preprocessing component in `sklearn`, implement a corresponding class that includes:
1. Inheritance to `BaseEstimator` and `TransformerMixin`
2. The method `fit`
3. The method `transform`

In [164]:
class StandardScaler:
    """
    Standard Scaler dari scratch.
    Formula: X_scaled = (X - mean) / std
    """
    def __init__(self):
        self.mean_ = None
        self.std_ = None
    
    def fit(self, X):
        """Hitung mean dan std dari training data"""
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)
        # Hindari pembagian dengan 0
        self.std_[self.std_ == 0] = 1
        return self
    
    def transform(self, X):
        """Transform data menggunakan mean dan std yang sudah di-fit"""
        if self.mean_ is None or self.std_ is None:
            raise ValueError("Scaler belum di-fit!")
        return (X - self.mean_) / self.std_
    
    def fit_transform(self, X):
        """Fit dan transform sekaligus"""
        return self.fit(X).transform(X)

### I. Feature Scaling

**Feature scaling** is a preprocessing technique used in machine learning to standardize the range of independent variables or features of data. The primary goal of feature scaling is to ensure that all features contribute equally to the training process and that machine learning algorithms can work effectively with the data.

Here are the main reasons why feature scaling is important:

1. **Algorithm Sensitivity:** Many machine learning algorithms are sensitive to the scale of input features. If the scales of features are significantly different, some algorithms may perform poorly or take much longer to converge.

2. **Distance-Based Algorithms:** Algorithms that rely on distances or similarities between data points, such as k-nearest neighbors (KNN) and support vector machines (SVM), can be influenced by feature scales. Features with larger scales may dominate the distance calculations.

3. **Regularization:** Regularization techniques, like L1 (Lasso) and L2 (Ridge) regularization, add penalty terms based on feature coefficients. Scaling ensures that all features are treated equally in the regularization process.

Common methods for feature scaling include:

1. **Min-Max Scaling (Normalization):** This method scales features to a specific range, typically [0, 1]. It's done using the following formula:

   $$X' = \frac{X - X_{min}}{X_{max} - X_{min}}$$

   - Here, $X$ is the original feature value, $X_{min}$ is the minimum value of the feature, and $X_{max}$ is the maximum value of the feature.  
<br />
<br />
2. **Standardization (Z-score Scaling):** This method scales features to have a mean (average) of 0 and a standard deviation of 1. It's done using the following formula:

   $$X' = \frac{X - \mu}{\sigma}$$

   - $X$ is the original feature value, $\mu$ is the mean of the feature, and $\sigma$ is the standard deviation of the feature.  
<br />
<br />
3. **Robust Scaling:** Robust scaling is a method that scales features to the interquartile range (IQR) and is less affected by outliers. It's calculated as:

   $$X' = \frac{X - Q1}{Q3 - Q1}$$

   - $X$ is the original feature value, $Q1$ is the first quartile (25th percentile), and $Q3$ is the third quartile (75th percentile) of the feature.  
<br />
<br />
4. **Log Transformation:** In cases where data is highly skewed or has a heavy-tailed distribution, taking the logarithm of the feature values can help stabilize the variance and improve scaling.

The choice of scaling method depends on the characteristics of your data and the requirements of your machine learning algorithm. **Min-max scaling and standardization are the most commonly used techniques and work well for many datasets.**

Scaling should be applied separately to each training and test set to prevent data leakage from the test set into the training set. Additionally, **some algorithms may not require feature scaling, particularly tree-based models.**

In [165]:
print("="*100)
print("FEATURE SCALING - SKIPPED FOR DECISION TREES")
print("="*100)

print("Decision Trees are SCALE-INVARIANT:")
print("  1. Tree splits based on thresholds (X > threshold), not distances")
print("  2. Scaling doesn't change the order or relationship between values")
print("  3. Removing scaler reduces potential transformation errors")
print()
print("When scaling IS needed:")
print("  * Distance-based algorithms (KNN, SVM)")
print("  * Gradient-based algorithms (Logistic Regression, Neural Networks)")
print("  * Algorithms with regularization")
print()
print("Strategy: Skip scaling, use raw features after encoding")
print("-"*100)

# Skip scaling entirely - use features directly
X_train_scaled = X_train_fe.copy()
X_val_scaled = X_val_fe.copy()

print(f"\n[OK] Features ready without scaling")
print(f"  Total features: {X_train_scaled.shape[1]}")
print(f"  Numerical: {len([col for col in numerical_cols if col in X_train_scaled.columns])}")
print(f"  Categorical: {len([col for col in categorical_cols if col in X_train_scaled.columns])}")

print("\n" + "="*100)
print("[OK] SCALING SKIPPED - RAW FEATURES PRESERVED")
print("="*100)

FEATURE SCALING - SKIPPED FOR DECISION TREES
Decision Trees are SCALE-INVARIANT:
  1. Tree splits based on thresholds (X > threshold), not distances
  2. Scaling doesn't change the order or relationship between values
  3. Removing scaler reduces potential transformation errors

When scaling IS needed:
  * Distance-based algorithms (KNN, SVM)
  * Gradient-based algorithms (Logistic Regression, Neural Networks)
  * Algorithms with regularization

Strategy: Skip scaling, use raw features after encoding
----------------------------------------------------------------------------------------------------

[OK] Features ready without scaling
  Total features: 44
  Numerical: 25
  Categorical: 18

[OK] SCALING SKIPPED - RAW FEATURES PRESERVED


**Penjelasan Feature Scaling:**

Standardization mengubah data sehingga memiliki mean = 0 dan standard deviation = 1.

**Kenapa perlu scaling?**
- Algoritma seperti SVM dan Logistic Regression menggunakan distance-based calculations
- Feature dengan nilai besar (misal: GDP dalam triliunan) akan mendominasi feature dengan nilai kecil (misal: approval rate 0-1)
- Scaling membuat semua feature berkontribusi seimbang

**Decision Tree tidak butuh scaling**, tapi kita tetap lakukan untuk konsistensi dengan model lain (LogReg, SVM).

### II. Feature Encoding

**Feature encoding**, also known as **categorical encoding**, is the process of converting categorical data (non-numeric data) into a numerical format so that it can be used as input for machine learning algorithms. Most machine learning models require numerical data for training and prediction, so feature encoding is a critical step in data preprocessing.

Categorical data can take various forms, including:

1. **Nominal Data:** Categories with no intrinsic order, like colors or country names.  

2. **Ordinal Data:** Categories with a meaningful order but not necessarily equidistant, like education levels (e.g., "high school," "bachelor's," "master's").

There are several common methods for encoding categorical data:

1. **Label Encoding:**

   - Label encoding assigns a unique integer to each category in a feature.
   - It's suitable for ordinal data where there's a clear order among categories.
   - For example, if you have an "education" feature with values "high school," "bachelor's," and "master's," you can encode them as 0, 1, and 2, respectively.
<br />
<br />
2. **One-Hot Encoding:**

   - One-hot encoding creates a binary (0 or 1) column for each category in a nominal feature.
   - It's suitable for nominal data where there's no inherent order among categories.
   - Each category becomes a new feature, and the presence (1) or absence (0) of a category is indicated for each row.
<br />
<br />
3. **Target Encoding (Mean Encoding):**

   - Target encoding replaces each category with the mean of the target variable for that category.
   - It's often used for classification problems.

In [166]:
class LabelEncoder:
    """
    Label Encoder dari scratch.
    Mengubah categorical values menjadi integer.
    """
    def __init__(self):
        self.label_mapping_ = {}
        self.inverse_mapping_ = {}
    
    def fit(self, X, columns):
        """Buat mapping dari categorical values ke integer"""
        for col in columns:
            if col not in X.columns:
                print(f"Warning: Column '{col}' not found in data. Skipping...")
                continue
            unique_values = X[col].unique()
            mapping = {val: idx for idx, val in enumerate(unique_values)}
            self.label_mapping_[col] = mapping
            self.inverse_mapping_[col] = {idx: val for val, idx in mapping.items()}
        return self
    
    def transform(self, X, columns):
        """Transform categorical values ke integer"""
        X_encoded = X.copy()
        for col in columns:
            if col not in X.columns:
                continue
            if col in self.label_mapping_:
                X_encoded[col] = X[col].map(self.label_mapping_[col])
                # Handle unseen values
                X_encoded[col] = X_encoded[col].fillna(-1).astype(int)
        return X_encoded
    
    def fit_transform(self, X, columns):
        """Fit dan transform sekaligus"""
        return self.fit(X, columns).transform(X, columns)

print("="*100)
print("FEATURE ENCODING")
print("="*100)

print("Label Encoding: Convert categorical values menjadi integer")
print("Contoh: 'Dropout' -> 0, 'Enrolled' -> 1, 'Graduate' -> 2")
print("-"*100)

# Filter categorical_cols to only include columns that exist
existing_categorical_cols = [col for col in categorical_cols if col in X_train_scaled.columns]
if len(existing_categorical_cols) < len(categorical_cols):
    missing_cols = set(categorical_cols) - set(existing_categorical_cols)
    print(f"\nWarning: {len(missing_cols)} categorical columns not found in data:")
    for col in missing_cols:
        print(f"  - {col}")
    print()

# Encode categorical features
label_encoder = LabelEncoder()
X_train_encoded = label_encoder.fit_transform(X_train_scaled, existing_categorical_cols)
X_val_encoded = label_encoder.transform(X_val_scaled, existing_categorical_cols)

print(f"\nCategorical features encoded: {len(existing_categorical_cols)}")
print("Sample encoding mappings:")
for i, col in enumerate(list(label_encoder.label_mapping_.keys())[:3]):
    print(f"  {col}: {len(label_encoder.label_mapping_[col])} unique values")

# Update categorical_cols to only existing columns
categorical_cols = existing_categorical_cols

# Encode target
target_encoder = LabelEncoder()
y_train_encoded = pd.Series(
    target_encoder.fit_transform(pd.DataFrame({'Target': y_train}), ['Target'])['Target'].values,
    index=y_train.index
)
y_val_encoded = pd.Series(
    target_encoder.transform(pd.DataFrame({'Target': y_val}), ['Target'])['Target'].values,
    index=y_val.index
)

print(f"\nTarget encoding:")
for val, idx in target_encoder.label_mapping_['Target'].items():
    print(f"  '{val}' -> {idx}")

print("\n" + "="*100)
print("ENCODING SELESAI")
print("="*100)

FEATURE ENCODING
Label Encoding: Convert categorical values menjadi integer
Contoh: 'Dropout' -> 0, 'Enrolled' -> 1, 'Graduate' -> 2
----------------------------------------------------------------------------------------------------

  - Nationality


Categorical features encoded: 18
Sample encoding mappings:
  Marital status: 6 unique values
  Application mode: 18 unique values
  Application order: 8 unique values

Target encoding:
  'Dropout' -> 0
  'Enrolled' -> 1
  'Graduate' -> 2

ENCODING SELESAI


**Penjelasan Label Encoding:**

Machine learning models hanya bisa memproses angka, tidak bisa memproses text/string. Label encoding mengubah categorical values menjadi integer.

**Contoh:**
- Marital status: 'Single' -> 0, 'Married' -> 1, 'Divorced' -> 2
- Target: 'Dropout' -> 0, 'Enrolled' -> 1, 'Graduate' -> 2

**Mengapa bukan One-Hot Encoding?**
- One-hot encoding membuat banyak kolom baru (curse of dimensionality)
- Untuk Decision Tree, label encoding sudah cukup
- Tree-based models bisa handle ordinal relationship dengan baik

### III. Handling Imbalanced Dataset (SKIPPED)

**Handling imbalanced datasets** is important because imbalanced data can lead to several issues that negatively impact the performance and reliability of machine learning models.

However, **untuk Decision Tree kami skip langkah ini** karena:

1. **Decision Tree secara natural robust terhadap imbalanced data**
   - Tree-based models membuat split berdasarkan information gain/gini impurity
   - Tidak seperti distance-based models yang sensitif terhadap class distribution

2. **Class distribution di dataset ini relatif balanced**
   - Dropout, Enrolled, dan Graduate memiliki proporsi yang cukup seimbang
   - Tidak ada kelas yang terlalu dominan (>80%)

3. **Resampling bisa introduce bias**
   - Oversampling: Risiko overfitting pada synthetic data
   - Undersampling: Kehilangan informasi penting

**Strategi alternatif yang bisa digunakan:**
- Adjust class weights dalam model (jika ada)
- Gunakan stratified sampling saat split data (sudah dilakukan)
- Gunakan evaluation metric yang appropriate (precision, recall, F1-score)

In [167]:
print("="*100)
print("IMBALANCED DATASET HANDLING - SKIPPED FOR DECISION TREE")
print("="*100)

print("Checking class distribution:")
print("\nTraining set:")
print(y_train.value_counts())
print("\nTraining set (percentage):")
print(y_train.value_counts(normalize=True) * 100)

print("\nValidation set:")
print(y_val.value_counts())
print("\nValidation set (percentage):")
print(y_val.value_counts(normalize=True) * 100)

print("\n" + "-"*100)
print("KESIMPULAN:")
print("- Class distribution relatif balanced (tidak ada kelas > 50%)")
print("- Decision Tree robust terhadap imbalanced data")
print("- Tidak perlu resampling (SMOTE/undersampling)")
print("- Sudah menggunakan stratified split untuk maintain distribution")
print("="*100)

IMBALANCED DATASET HANDLING - SKIPPED FOR DECISION TREE
Checking class distribution:

Training set:
Target
Graduate    1236
Dropout      795
Enrolled     445
Name: count, dtype: int64

Training set (percentage):
Target
Graduate    49.919225
Dropout     32.108239
Enrolled    17.972536
Name: proportion, dtype: float64

Validation set:
Target
Graduate    310
Dropout     199
Enrolled    111
Name: count, dtype: int64

Validation set (percentage):
Target
Graduate    50.000000
Dropout     32.096774
Enrolled    17.903226
Name: proportion, dtype: float64

----------------------------------------------------------------------------------------------------
KESIMPULAN:
- Class distribution relatif balanced (tidak ada kelas > 50%)
- Decision Tree robust terhadap imbalanced data
- Tidak perlu resampling (SMOTE/undersampling)
- Sudah menggunakan stratified split untuk maintain distribution


### IV. Data Normalization (SKIPPED)

Data normalization is used to achieve a standard distribution (typically normal/Gaussian distribution). 

However, **untuk Decision Tree kami skip langkah ini** karena:

1. **Decision Tree tidak memerlukan data terdistribusi normal**
   - Tree splits berdasarkan threshold, bukan distribusi statistik
   - Tidak ada asumsi tentang distribusi data

2. **Sudah melakukan Standardization**
   - Standardization (mean=0, std=1) sudah cukup untuk keperluan preprocessing
   - Normalization (transform ke distribusi normal) berbeda dengan standardization

3. **Normalization lebih penting untuk models tertentu**
   - Gaussian Naive Bayes (asumsi normal distribution)
   - Beberapa neural networks

**Catatan:** Standardization != Normalization
- Standardization: Scale data (mean=0, std=1)
- Normalization: Transform distribution menjadi normal (misalnya dengan Box-Cox)

In [168]:
print("="*100)
print("DATA NORMALIZATION - SKIPPED FOR DECISION TREE")
print("="*100)

print("Decision Tree tidak memerlukan data terdistribusi normal")
print("Tree-based models membuat split berdasarkan threshold, bukan distribusi")
print()
print("Standardization (sudah dilakukan) sudah cukup untuk preprocessing")
print("="*100)

DATA NORMALIZATION - SKIPPED FOR DECISION TREE
Decision Tree tidak memerlukan data terdistribusi normal
Tree-based models membuat split berdasarkan threshold, bukan distribusi

Standardization (sudah dilakukan) sudah cukup untuk preprocessing


### V. Dimensionality Reduction - SKIPPED for Decision Trees

**Why skip PCA for Decision Trees?**

Decision Trees are **non-linear models** that don't require dimensionality reduction:

1. **Handle High Dimensions Well**: Trees naturally select important features via information gain
2. **Feature Interpretability**: Original features lebih interpretable daripada principal components
3. **No Linearity Assumption**: Trees capture non-linear patterns tanpa perlu transform linear seperti PCA
4. **Information Loss**: PCA dengan 95% variance tetap hilangkan 5% informasi yang mungkin penting

**When to use PCA?**
- Linear models (Logistic Regression, SVM) -> benefit from reduced dimensions
- Neural Networks -> faster training
- High multicollinearity -> decorrelate features

**Our strategy:** Skip PCA, use all features, let tree naturally select the best ones

In [169]:
print("="*100)
print("DIMENSIONALITY REDUCTION - SKIPPED FOR DECISION TREES")
print("="*100)

print("Decision Trees TIDAK memerlukan PCA karena:")
print("  1. Naturally select important features via information gain")
print("  2. Original features lebih interpretable")
print("  3. Capture non-linear patterns without linear transform")
print("  4. PCA 95% variance = loss 5% information yang mungkin penting")
print()
print("Strategy: Use ALL features, let tree do feature selection")
print("-"*100)

# Combine numerical and categorical features WITHOUT PCA
X_train_final = X_train_encoded.values
X_val_final = X_val_encoded.values

print(f"\nFinal feature count: {X_train_final.shape[1]}")
print(f"  Numerical features: {len([col for col in numerical_cols if col in X_train_encoded.columns])}")
print(f"  Categorical features: {len(categorical_cols)}")

print("\n" + "="*100)
print("[OK] ALL FEATURES READY (No PCA)")
print("="*100)

DIMENSIONALITY REDUCTION - SKIPPED FOR DECISION TREES
Decision Trees TIDAK memerlukan PCA karena:
  1. Naturally select important features via information gain
  2. Original features lebih interpretable
  3. Capture non-linear patterns without linear transform
  4. PCA 95% variance = loss 5% information yang mungkin penting

Strategy: Use ALL features, let tree do feature selection
----------------------------------------------------------------------------------------------------

Final feature count: 44
  Numerical features: 25
  Categorical features: 18

[OK] ALL FEATURES READY (No PCA)


**Penjelasan PCA (Principal Component Analysis):**

PCA adalah teknik dimensionality reduction yang mengurangi jumlah features dengan tetap mempertahankan informasi penting.

**Cara kerja:**
1. Cari arah (principal components) di mana data memiliki variance terbesar
2. Project data ke principal components tersebut
3. Pilih sejumlah components yang menjelaskan 95% variance

**Keuntungan:**
- Mengurangi computational cost
- Mengurangi risk of overfitting
- Menghilangkan multicollinearity (features yang berkorelasi tinggi)

**Catatan:**
- PCA hanya diterapkan pada **numerical features**
- Categorical features tetap dipertahankan (tidak di-transform)
- Loss of interpretability: Principal components tidak punya makna langsung

**Alternatif jika tidak ingin PCA:**
- Feature selection dengan Chi-Square test (categorical)
- Feature selection dengan ANOVA F-test (numerical)
- Tree-based feature importance

# 3. Compile Preprocessing Pipeline

All of the preprocessing classes or functions defined earlier will be compiled in this step.

If you use sklearn to create preprocessing classes, you can list your preprocessing classes in the Pipeline object sequentially, and then fit and transform your data.

In [170]:
# from sklearn.pipeline import Pipeline

# # Note: You can add or delete preprocessing components from this pipeline

# pipe = Pipeline([("imputer", FeatureImputer()),
#                  ("featurecreator", FeatureCreator()),
#                  ("scaler", FeatureScaler()),
#                  ("encoder", FeatureEncoder())])

# train_set = pipe.fit_transform(train_set)
# val_set = pipe.transform(val_set)

In [171]:
# # Your code should work up until this point
# train_set = pipe.fit_transform(train_set)
# val_set = pipe.transform(val_set)

or create your own here

In [172]:
# Write your code here

# 4. Modeling and Validation

Modelling is the process of building your own machine learning models to solve specific problems, or in this assignment context, predicting the target feature `attack_cat`. Validation is the process of evaluating your trained model using the validation set or cross-validation method and providing some metrics that can help you decide what to do in the next iteration of development.

## A. DTL

In [173]:
class SMOTE:
    """
    Synthetic Minority Over-sampling Technique (SMOTE)
    
    Generate synthetic samples untuk minority class dengan:
    1. Pilih random sample dari minority class
    2. Cari k nearest neighbors
    3. Pilih random neighbor
    4. Generate synthetic sample di antara sample dan neighbor
    
    Parameters:
    -----------
    k_neighbors : int
        Number of nearest neighbors untuk generate synthetic samples
    sampling_strategy : float or 'auto'
        Ratio minoritas/mayoritas yang diinginkan
    """
    def __init__(self, k_neighbors=5, sampling_strategy='auto', random_state=42):
        self.k_neighbors = k_neighbors
        self.sampling_strategy = sampling_strategy
        self.random_state = random_state
    
    def _euclidean_distance(self, x1, x2):
        """Calculate Euclidean distance"""
        return np.sqrt(np.sum((x1 - x2) ** 2))
    
    def _get_neighbors(self, X, sample_idx, k):
        """Get k nearest neighbors untuk sample"""
        sample = X[sample_idx]
        distances = []
        
        for idx in range(len(X)):
            if idx != sample_idx:
                dist = self._euclidean_distance(sample, X[idx])
                distances.append((idx, dist))
        
        # Sort by distance dan ambil k terdekat
        distances.sort(key=lambda x: x[1])
        neighbors = [idx for idx, _ in distances[:k]]
        return neighbors
    
    def fit_resample(self, X, y):
        """
        Generate synthetic samples untuk minority classes.
        
        Returns:
        --------
        X_resampled, y_resampled : arrays
            Resampled data dengan synthetic samples
        """
        np.random.seed(self.random_state)
        
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values
        
        # Hitung class distribution
        classes, counts = np.unique(y, return_counts=True)
        max_count = np.max(counts)
        
        print(f"\\nOriginal class distribution:")
        for cls, count in zip(classes, counts):
            print(f"  Class {cls}: {count} samples ({count/len(y)*100:.1f}%)")
        
        X_resampled = X.copy()
        y_resampled = y.copy()
        
        # Generate synthetic samples untuk setiap minority class
        for cls, count in zip(classes, counts):
            if count < max_count:
                # Tentukan berapa synthetic samples yang perlu di-generate
                if self.sampling_strategy == 'auto':
                    n_synthetic = max_count - count
                else:
                    n_synthetic = int(max_count * self.sampling_strategy) - count
                
                if n_synthetic <= 0:
                    continue
                
                # Get samples dari class ini
                class_indices = np.where(y == cls)[0]
                X_class = X[class_indices]
                
                synthetic_samples = []
                
                # Generate synthetic samples
                for _ in range(n_synthetic):
                    # Pilih random sample
                    sample_idx = np.random.randint(0, len(X_class))
                    
                    # Get k neighbors
                    k = min(self.k_neighbors, len(X_class) - 1)
                    if k <= 0:
                        continue
                    
                    neighbors = self._get_neighbors(X_class, sample_idx, k)
                    
                    # Pilih random neighbor
                    neighbor_idx = neighbors[np.random.randint(0, len(neighbors))]
                    
                    # Generate synthetic sample
                    sample = X_class[sample_idx]
                    neighbor = X_class[neighbor_idx]
                    alpha = np.random.random()
                    synthetic = sample + alpha * (neighbor - sample)
                    
                    synthetic_samples.append(synthetic)
                
                # Add synthetic samples
                if synthetic_samples:
                    X_resampled = np.vstack([X_resampled, synthetic_samples])
                    y_resampled = np.concatenate([y_resampled, [cls] * len(synthetic_samples)])
                    
                    print(f"  Generated {len(synthetic_samples)} synthetic samples for class {cls}")
        
        print(f"\\nResampled class distribution:")
        classes, counts = np.unique(y_resampled, return_counts=True)
        for cls, count in zip(classes, counts):
            print(f"  Class {cls}: {count} samples ({count/len(y_resampled)*100:.1f}%)")
        
        print(f"\\nTotal samples: {len(y)} -> {len(y_resampled)} (+{len(y_resampled)-len(y)})")
        
        return X_resampled, y_resampled

print("="*100)
print("SMOTE (Synthetic Minority Over-sampling Technique)")
print("="*100)
print("Technique untuk handle class imbalance dengan generate synthetic samples")
print("="*100)

SMOTE (Synthetic Minority Over-sampling Technique)
Technique untuk handle class imbalance dengan generate synthetic samples


In [174]:
class DecisionTreeNode:
    """
    Node structure untuk Decision Tree.
    Setiap node merepresentasikan decision point atau leaf.
    """
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None, samples=0, impurity=0):
        self.feature = feature          # Index feature untuk split
        self.threshold = threshold      # Threshold value untuk continuous feature
        self.left = left               # Left child (<=threshold)
        self.right = right             # Right child (>threshold)
        self.value = value             # Class prediction untuk leaf node
        self.samples = samples         # Jumlah samples di node ini
        self.impurity = impurity       # Impurity value untuk pruning

class DecisionTreeClassifier:
    """
    Decision Tree Classifier dengan CART algorithm + Cost-Complexity Pruning.
    
    Enhancements:
    - Class weights untuk handle imbalanced data
    - Cost-complexity pruning (post-pruning)
    - Feature importance tracking
    - Support Gini dan Entropy criterion
    
    Parameters:
    -----------
    max_depth : int
        Maximum depth of tree
    min_samples_split : int
        Minimum samples required untuk split internal node
    min_samples_leaf : int
        Minimum samples required untuk leaf node
    criterion : str
        'gini' atau 'entropy' untuk splitting criterion
    class_weight : dict or 'balanced'
        Weights untuk setiap class. 'balanced' otomatis calculate weights
    ccp_alpha : float
        Cost-complexity parameter untuk pruning. 0.0 = no pruning
    """
    def __init__(self, max_depth=10, min_samples_split=2, min_samples_leaf=1, 
                 criterion='gini', class_weight=None, ccp_alpha=0.0):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.class_weight = class_weight
        self.ccp_alpha = ccp_alpha
        self.tree = None
        self.n_classes = None
        self.n_features = None
        self.feature_importances_ = None
        self.class_weights_ = None
    
    def _calculate_entropy(self, y):
        """
        Hitung entropy dari label distribution.
        
        Entropy = -Σ(p_i * log2(p_i))
        
        Entropy mengukur "ketidakpastian" atau "impurity" dari data:
        - Entropy = 0: semua samples dalam 1 class (pure)
        - Entropy maksimal: samples terdistribusi merata di semua class
        
        Returns:
        --------
        float : entropy value
        """
        if len(y) == 0:
            return 0
        
        # Hitung proporsi setiap class
        _, counts = np.unique(y, return_counts=True)
        proportions = counts / len(y)
        
        # Hitung entropy
        entropy = -np.sum([p * np.log2(p) for p in proportions if p > 0])
        return entropy
    
    def _calculate_gini(self, y):
        """
        Hitung Gini Impurity dari label distribution.
        
        Gini = 1 - Σ(p_i²)
        
        Gini impurity mengukur probability misclassification:
        - Gini = 0: pure node (1 class)
        - Gini maksimal ~= 0.5: balanced distribution (binary)
        
        Returns:
        --------
        float : gini impurity value
        """
        if len(y) == 0:
            return 0
        
        # Hitung proporsi setiap class
        _, counts = np.unique(y, return_counts=True)
        proportions = counts / len(y)
        
        # Hitung gini impurity
        gini = 1 - np.sum(proportions ** 2)
        return gini
    
    def _calculate_impurity(self, y):
        """
        Wrapper function untuk hitung impurity berdasarkan criterion.
        """
        if self.criterion == 'entropy':
            return self._calculate_entropy(y)
        else:  # gini
            return self._calculate_gini(y)
    
    def _find_best_continuous_split(self, X_column, y):
        """
        Cari best threshold untuk continuous feature.
        
        Strategi C4.5-like:
        - Sort unique values
        - Test midpoint antara consecutive values sebagai threshold
        - Pilih threshold dengan information gain tertinggi
        
        Parameters:
        -----------
        X_column : array
            Feature values untuk 1 kolom
        y : array
            Target labels
            
        Returns:
        --------
        best_threshold : float
            Best threshold value
        best_gain : float
            Information gain dari split
        """
        best_gain = -1
        best_threshold = None
        
        # Get sorted unique values
        unique_values = np.unique(X_column)
        
        # Jika hanya 1 unique value, tidak bisa split
        if len(unique_values) <= 1:
            return None, -1
        
        # Test midpoint antara consecutive values
        for i in range(len(unique_values) - 1):
            threshold = (unique_values[i] + unique_values[i + 1]) / 2
            
            # Split data
            left_mask = X_column <= threshold
            right_mask = ~left_mask
            
            y_left = y[left_mask]
            y_right = y[right_mask]
            
            # Check minimum samples
            if len(y_left) < self.min_samples_leaf or len(y_right) < self.min_samples_leaf:
                continue
            
            # Hitung information gain
            gain = self._information_gain(y, y_left, y_right)
            
            if gain > best_gain:
                best_gain = gain
                best_threshold = threshold
        
        return best_threshold, best_gain
    
    def _split_data(self, X, y, feature_idx, threshold):
        """
        Split data berdasarkan feature dan threshold.
        
        Parameters:
        -----------
        X : array
            Feature matrix
        y : array
            Target labels
        feature_idx : int
            Index feature untuk split
        threshold : float
            Threshold value
            
        Returns:
        --------
        X_left, X_right, y_left, y_right : arrays
            Split data
        """
        left_mask = X[:, feature_idx] <= threshold
        right_mask = ~left_mask
        
        return (X[left_mask], X[right_mask], 
                y[left_mask], y[right_mask])
    
    def _information_gain(self, y_parent, y_left, y_right):
        """
        Hitung information gain dari split.
        
        Information Gain = Impurity(parent) - Weighted_Impurity(children)
        
        Weighted_Impurity = (n_left/n) * Impurity(left) + (n_right/n) * Impurity(right)
        
        Parameters:
        -----------
        y_parent : array
            Parent node labels
        y_left : array
            Left child labels
        y_right : array
            Right child labels
            
        Returns:
        --------
        float : information gain value
        """
        parent_impurity = self._calculate_impurity(y_parent)
        n = len(y_parent)
        n_left, n_right = len(y_left), len(y_right)
        
        # Avoid division by zero
        if n_left == 0 or n_right == 0:
            return 0
        
        # Weighted impurity of children
        weighted_impurity = (n_left / n) * self._calculate_impurity(y_left) + \
                           (n_right / n) * self._calculate_impurity(y_right)
        
        # Information gain
        gain = parent_impurity - weighted_impurity
        return gain
    
    def _find_best_split(self, X, y):
        """
        Cari best split across all features.
        
        Untuk setiap feature:
        1. Cari best threshold (untuk continuous)
        2. Hitung information gain
        3. Pilih feature + threshold dengan gain tertinggi
        
        Parameters:
        -----------
        X : array
            Feature matrix
        y : array
            Target labels
            
        Returns:
        --------
        best_feature : int
            Index feature terbaik
        best_threshold : float
            Threshold value terbaik
        """
        best_gain = -1
        best_feature = None
        best_threshold = None
        
        n_features = X.shape[1]
        
        # Loop through all features
        for feature_idx in range(n_features):
            X_column = X[:, feature_idx]
            
            # Find best split untuk feature ini
            threshold, gain = self._find_best_continuous_split(X_column, y)
            
            if gain > best_gain:
                best_gain = gain
                best_feature = feature_idx
                best_threshold = threshold
        
        return best_feature, best_threshold
    
    def _majority_class(self, y):
        """
        Return majority class dari labels.
        
        Parameters:
        -----------
        y : array
            Target labels
            
        Returns:
        --------
        int : majority class
        """
        if len(y) == 0:
            return 0
        return np.bincount(y).argmax()
    
    def _build_tree(self, X, y, depth=0):
        """
        Build tree secara rekursif.
        
        Algoritma:
        1. Check stopping criteria
        2. Find best split
        3. Split data
        4. Recursively build left dan right subtrees
        
        Parameters:
        -----------
        X : array
            Feature matrix
        y : array
            Target labels
        depth : int
            Current depth
            
        Returns:
        --------
        DecisionTreeNode : root node of (sub)tree
        """
        n_samples = len(y)
        n_classes = len(np.unique(y))
        
        # Stopping criteria
        # 1. Reached max depth
        # 2. Not enough samples to split
        # 3. Pure node (all same class)
        if (depth >= self.max_depth or 
            n_samples < self.min_samples_split or 
            n_classes == 1):
            leaf_value = self._majority_class(y)
            return DecisionTreeNode(value=leaf_value, samples=n_samples)
        
        # Find best split
        best_feature, best_threshold = self._find_best_split(X, y)
        
        # If no valid split found, create leaf
        if best_feature is None:
            leaf_value = self._majority_class(y)
            return DecisionTreeNode(value=leaf_value, samples=n_samples)
        
        # Split data
        X_left, X_right, y_left, y_right = self._split_data(
            X, y, best_feature, best_threshold
        )
        
        # Build subtrees recursively
        left_subtree = self._build_tree(X_left, y_left, depth + 1)
        right_subtree = self._build_tree(X_right, y_right, depth + 1)
        
        # Create internal node
        return DecisionTreeNode(
            feature=best_feature,
            threshold=best_threshold,
            left=left_subtree,
            right=right_subtree,
            samples=n_samples
        )
    
    def _calculate_class_weights(self, y):
        """
        Calculate class weights untuk handle imbalanced data.
        
        Formula: weight[class] = n_samples / (n_classes * n_samples_class)
        """
        classes, counts = np.unique(y, return_counts=True)
        n_samples = len(y)
        n_classes = len(classes)
        
        weights = {}
        for cls, count in zip(classes, counts):
            weights[cls] = n_samples / (n_classes * count)
        
        return weights
    
    def _get_sample_weights(self, y):
        """
        Get sample weights berdasarkan class weights.
        """
        if self.class_weights_ is None:
            return np.ones(len(y))
        
        sample_weights = np.array([self.class_weights_[label] for label in y])
        return sample_weights
    
    def _weighted_impurity(self, y, sample_weights=None):
        """
        Hitung weighted impurity.
        """
        if sample_weights is None:
            return self._calculate_impurity(y)
        
        if len(y) == 0:
            return 0
        
        # Weighted class proportions
        classes = np.unique(y)
        total_weight = np.sum(sample_weights)
        
        if self.criterion == 'entropy':
            entropy = 0
            for cls in classes:
                mask = y == cls
                p = np.sum(sample_weights[mask]) / total_weight
                if p > 0:
                    entropy -= p * np.log2(p)
            return entropy
        else:  # gini
            gini = 1
            for cls in classes:
                mask = y == cls
                p = np.sum(sample_weights[mask]) / total_weight
                gini -= p ** 2
            return gini
    
    def _calculate_feature_importance(self, node, feature_importances, n_samples_total):
        """
        Calculate feature importance recursively.
        
        Importance = (n_samples / n_total) * (impurity - weighted_impurity_children)
        """
        if node is None or node.value is not None:
            return
        
        # Calculate importance contribution
        if node.feature is not None:
            importance = (node.samples / n_samples_total) * (
                node.impurity - 
                (node.left.samples / node.samples) * node.left.impurity -
                (node.right.samples / node.samples) * node.right.impurity
            )
            feature_importances[node.feature] += importance
        
        # Recurse
        self._calculate_feature_importance(node.left, feature_importances, n_samples_total)
        self._calculate_feature_importance(node.right, feature_importances, n_samples_total)
    
    def _prune_tree(self, node, alpha):
        """
        Cost-Complexity Pruning (simplified version).
        
        Prune nodes where cost of leaf is better than cost of subtree.
        This version doesn't need to pass X and y recursively.
        """
        if node is None or node.value is not None:
            return node
        
        # Recursively prune children first
        if node.left is not None:
            node.left = self._prune_tree(node.left, alpha)
        if node.right is not None:
            node.right = self._prune_tree(node.right, alpha)
        
        # If both children are leaves, consider pruning this node
        if (node.left is not None and node.left.value is not None and 
            node.right is not None and node.right.value is not None):
            
            # Calculate complexity cost
            # Simple heuristic: if node has very few samples compared to children,
            # or if alpha is high enough, consider pruning
            
            # Number of leaves in subtree
            n_leaves_subtree = 2
            
            # Weighted error reduction from split
            # If impurity reduction is small, prune
            impurity_reduction = (node.impurity - 
                                 (node.left.samples / node.samples) * node.left.impurity -
                                 (node.right.samples / node.samples) * node.right.impurity)
            
            # Prune if complexity cost outweighs benefit
            if impurity_reduction < alpha * n_leaves_subtree:
                # Convert to leaf node with majority class
                # Use majority from left and right
                if node.left.samples > node.right.samples:
                    leaf_value = node.left.value
                else:
                    leaf_value = node.right.value
                
                return DecisionTreeNode(
                    value=leaf_value, 
                    samples=node.samples, 
                    impurity=node.impurity
                )
        
        return node
    
    def _build_tree(self, X, y, sample_weights=None, depth=0):
        """
        Build tree secara rekursif dengan support untuk weighted samples.
        """
        n_samples = len(y)
        n_classes = len(np.unique(y))
        
        # Calculate impurity
        if sample_weights is None:
            impurity = self._calculate_impurity(y)
        else:
            impurity = self._weighted_impurity(y, sample_weights)
        
        # Stopping criteria
        if (depth >= self.max_depth or 
            n_samples < self.min_samples_split or 
            n_classes == 1):
            leaf_value = self._majority_class(y)
            return DecisionTreeNode(value=leaf_value, samples=n_samples, impurity=impurity)
        
        # Find best split
        best_feature, best_threshold = self._find_best_split(X, y)
        
        # If no valid split found, create leaf
        if best_feature is None:
            leaf_value = self._majority_class(y)
            return DecisionTreeNode(value=leaf_value, samples=n_samples, impurity=impurity)
        
        # Split data
        X_left, X_right, y_left, y_right = self._split_data(
            X, y, best_feature, best_threshold
        )
        
        # Split sample weights
        if sample_weights is not None:
            left_mask = X[:, best_feature] <= best_threshold
            weights_left = sample_weights[left_mask]
            weights_right = sample_weights[~left_mask]
        else:
            weights_left = None
            weights_right = None
        
        # Build subtrees recursively
        left_subtree = self._build_tree(X_left, y_left, weights_left, depth + 1)
        right_subtree = self._build_tree(X_right, y_right, weights_right, depth + 1)
        
        # Create internal node
        return DecisionTreeNode(
            feature=best_feature,
            threshold=best_threshold,
            left=left_subtree,
            right=right_subtree,
            samples=n_samples,
            impurity=impurity
        )
    
    def fit(self, X, y):
        """
        Train decision tree with optional class weighting and pruning.
        """
        # Convert to numpy arrays
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values
        
        self.n_classes = len(np.unique(y))
        self.n_features = X.shape[1]
        
        # Calculate class weights
        if self.class_weight == 'balanced':
            self.class_weights_ = self._calculate_class_weights(y)
            print("\\nClass weights (balanced):")
            for cls, weight in self.class_weights_.items():
                print(f"  Class {cls}: {weight:.4f}")
        elif isinstance(self.class_weight, dict):
            self.class_weights_ = self.class_weight
            print("\\nUsing provided class weights:")
            for cls, weight in self.class_weights_.items():
                print(f"  Class {cls}: {weight:.4f}")
        else:
            self.class_weights_ = None
        
        # Get sample weights
        sample_weights = self._get_sample_weights(y) if self.class_weights_ else None
        
        # Build tree
        self.tree = self._build_tree(X, y, sample_weights)
        
        # Apply cost-complexity pruning
        if self.ccp_alpha > 0:
            print(f"\\nApplying cost-complexity pruning (alpha={self.ccp_alpha})...")
            original_leaves = self._count_leaves(self.tree)
            self.tree = self._prune_tree(self.tree, self.ccp_alpha)
            pruned_leaves = self._count_leaves(self.tree)
            print(f"  Leaves: {original_leaves} -> {pruned_leaves} (pruned {original_leaves - pruned_leaves})")
        
        # Calculate feature importances
        self.feature_importances_ = np.zeros(self.n_features)
        self._calculate_feature_importance(self.tree, self.feature_importances_, len(y))
        
        # Normalize feature importances
        if np.sum(self.feature_importances_) > 0:
            self.feature_importances_ /= np.sum(self.feature_importances_)
        
        return self
    
    def _predict_sample(self, x, node):
        """
        Prediksi untuk single sample dengan traversal tree.
        
        Parameters:
        -----------
        x : array
            Single sample
        node : DecisionTreeNode
            Current node
            
        Returns:
        --------
        int : predicted class
        """
        # If leaf node, return value
        if node.value is not None:
            return node.value
        
        # Otherwise, traverse tree
        if x[node.feature] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)
    
    def predict(self, X):
        """
        Predict class untuk samples.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Samples
            
        Returns:
        --------
        y : array, shape (n_samples,)
            Predicted classes
        """
        if isinstance(X, pd.DataFrame):
            X = X.values
        
        predictions = np.array([self._predict_sample(x, self.tree) for x in X])
        return predictions
    
    def _get_tree_depth(self, node):
        """
        Hitung depth tree secara rekursif.
        """
        if node is None or node.value is not None:
            return 0
        return 1 + max(self._get_tree_depth(node.left), 
                       self._get_tree_depth(node.right))
    
    def _count_leaves(self, node):
        """
        Hitung jumlah leaf nodes.
        """
        if node is None:
            return 0
        if node.value is not None:
            return 1
        return self._count_leaves(node.left) + self._count_leaves(node.right)
    
    def _get_used_features(self, node, features_set=None):
        """
        Get set of all features yang digunakan di tree.
        """
        if features_set is None:
            features_set = set()
        
        if node is None or node.value is not None:
            return features_set
        
        # Add current node's feature
        if node.feature is not None:
            features_set.add(node.feature)
        
        # Recurse to children
        self._get_used_features(node.left, features_set)
        self._get_used_features(node.right, features_set)
        
        return features_set
    
    def get_tree_info(self):
        """
        Get informasi tentang tree structure.
        """
        used_features = self._get_used_features(self.tree)
        
        return {
            'depth': self._get_tree_depth(self.tree),
            'n_leaves': self._count_leaves(self.tree),
            'n_features': self.n_features,
            'n_features_used': len(used_features),
            'used_features': sorted(list(used_features)),
            'n_classes': self.n_classes,
            'criterion': self.criterion,
            'ccp_alpha': self.ccp_alpha,
            'class_weighted': self.class_weights_ is not None
        }
    
    def get_feature_importances(self, feature_names=None):
        """
        Get feature importances dengan optional feature names.
        """
        if self.feature_importances_ is None:
            return None
        
        importances = []
        for idx, imp in enumerate(self.feature_importances_):
            name = feature_names[idx] if feature_names else f"Feature_{idx}"
            importances.append((name, imp))
        
        # Sort by importance
        importances.sort(key=lambda x: x[1], reverse=True)
        return importances
    
    def print_feature_importances(self, top_n=10, feature_names=None):
        """
        Print top N most important features.
        """
        importances = self.get_feature_importances(feature_names)
        if importances is None:
            print("Feature importances not available")
            return
        
        print(f"\\nTop {min(top_n, len(importances))} Feature Importances:")
        for i, (name, imp) in enumerate(importances[:top_n]):
            print(f"  {i+1}. {name}: {imp:.4f}")
    
    def print_tree(self, node=None, feature_names=None, depth=0, prefix="", is_left=True):
        """
        Print tree structure in text format.
        
        Parameters:
        -----------
        node : DecisionTreeNode
            Current node to print
        feature_names : list
            List of feature names
        depth : int
            Current depth
        prefix : str
            Prefix for indentation
        is_left : bool
            Whether this is left child
        """
        if node is None:
            node = self.tree
        
        if node.value is not None:
            # Leaf node
            print(f"{prefix}{'L' if is_left else 'R'}--- LEAF: Class={node.value} (samples={node.samples})")
            return
        
        # Internal node
        feature_name = feature_names[node.feature] if feature_names else f"X[{node.feature}]"
        print(f"{prefix}{'L' if is_left else 'R'}--- {feature_name} <= {node.threshold:.2f}? (samples={node.samples})")
        
        # Print left subtree (True branch)
        if node.left:
            extension = "|   " if node.right else "    "
            self.print_tree(node.left, feature_names, depth + 1, prefix + extension, True)
        
        # Print right subtree (False branch)
        if node.right:
            extension = "    "
            self.print_tree(node.right, feature_names, depth + 1, prefix + extension, False)
    
    def visualize_tree_summary(self, feature_names=None, max_depth_show=3):
        """
        Print concise tree summary showing structure up to max_depth_show.
        """
        print("\\n" + "="*100)
        print("TREE STRUCTURE VISUALIZATION")
        print("="*100)
        
        tree_info = self.get_tree_info()
        print(f"Tree Statistics:")
        print(f"  - Actual Depth: {tree_info['depth']}")
        print(f"  - Number of Leaves: {tree_info['n_leaves']}")
        print(f"  - Features Used: {tree_info['n_features_used']}/{tree_info['n_features']}")
        
        print(f"\\nTree Structure (showing depth 0-{max_depth_show}):")
        print("-"*100)
        self._print_tree_limited(self.tree, feature_names, 0, max_depth_show, "", True)
        
        if tree_info['depth'] > max_depth_show:
            print(f"\\n... (tree continues to depth {tree_info['depth']})")
        print("="*100)
    
    def _print_tree_limited(self, node, feature_names, depth, max_depth, prefix, is_left):
        """Helper function to print tree up to max_depth."""
        if node is None or depth > max_depth:
            if depth > max_depth and node.value is None:
                print(f"{prefix}{'L' if is_left else 'R'}--- ...")
            return
        
        if node.value is not None:
            print(f"{prefix}{'L' if is_left else 'R'}--- LEAF: Class={node.value} (n={node.samples})")
            return
        
        feature_name = feature_names[node.feature] if feature_names else f"X[{node.feature}]"
        print(f"{prefix}{'L' if is_left else 'R'}--- {feature_name} <= {node.threshold:.2f}?")
        
        if node.left:
            extension = "|   " if node.right else "    "
            self._print_tree_limited(node.left, feature_names, depth + 1, max_depth, prefix + extension, True)
        
        if node.right:
            extension = "    "
            self._print_tree_limited(node.right, feature_names, depth + 1, max_depth, prefix + extension, False)
    
    def save(self, filepath):
        """Save model ke file"""
        with open(filepath, 'wb') as f:
            pickle.dump(self, f)
    
    @staticmethod
    def load(filepath):
        """Load model dari file"""
        with open(filepath, 'rb') as f:
            return pickle.load(f)

print("="*100)
print("DECISION TREE CLASSIFIER (CART) - ENHANCED")
print("="*100)
print("Implementasi lengkap dengan:")
print("- Entropy dan Gini Impurity calculation")
print("- Continuous value splitting (C4.5-like)")
print("- Information Gain untuk feature selection")
print("- Class weighting untuk handle imbalanced data")
print("- Cost-Complexity Pruning (post-pruning)")
print("- Feature importance calculation")
print("- Binary tree structure")
print("="*100)

DECISION TREE CLASSIFIER (CART) - ENHANCED
Implementasi lengkap dengan:
- Entropy dan Gini Impurity calculation
- Continuous value splitting (C4.5-like)
- Information Gain untuk feature selection
- Class weighting untuk handle imbalanced data
- Cost-Complexity Pruning (post-pruning)
- Feature importance calculation
- Binary tree structure


## C. Improvements (Optional)

In [175]:
def calculate_metrics(y_true, y_pred, model_name):
    """Hitung accuracy, precision, recall, F1-score untuk setiap kelas"""
    from collections import Counter
    
    # Accuracy
    accuracy = np.mean(y_true == y_pred)
    
    # Per-class metrics
    classes = np.unique(y_true)
    precision_per_class = []
    recall_per_class = []
    f1_per_class = []
    
    for cls in classes:
        # True Positives, False Positives, False Negatives
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        
        # Precision, Recall, F1
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        precision_per_class.append(precision)
        recall_per_class.append(recall)
        f1_per_class.append(f1)
    
    print(f"\n{model_name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision per class: {[f'{p:.4f}' for p in precision_per_class]}")
    print(f"  Recall per class: {[f'{r:.4f}' for r in recall_per_class]}")
    print(f"  F1-score per class: {[f'{f:.4f}' for f in f1_per_class]}")
    
    return accuracy, precision_per_class, recall_per_class, f1_per_class

print("Metrics calculation function ready")

Metrics calculation function ready


### Strategy: Fokus pada Hyperparameter Optimization
Berdasarkan analisis:
- **SMOTE sering overfitting** pada small dataset dengan synthetic samples
- **Class weights** bisa bias model ke minority class
- **Fokus**: Find optimal tree structure (depth, split, leaf) dan criterion

Key insight: Decision Tree performance sangat tergantung pada structure parameters

In [176]:
print("\n" + "="*100)
print("QUICK DEPTH EXPLORATION - Find Best max_depth")
print("="*100)

# Test berbagai depth dengan fixed params yang reasonable
print("Strategy: Test different max_depth values quickly")
print("Fixed params: min_samples_split=20, min_samples_leaf=10, criterion=entropy")
print("\nNote:")
print("  - Total features available: 44 (original + engineered)")
print("  - Depth = max number of questions in a decision path")
print("  - Features used = how many different attributes tree actually needs")
print("  - Example: Depth=5 might only use 19/44 features (other 25 not important)")
print("-"*100)

depth_results = []
test_depths = [5, 8, 10, 12, 15, 18, 20, 25, 30]

# Get feature names for later
feature_names = list(X_train_encoded.columns)

for depth in test_depths:
    dt = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_split=20,
        min_samples_leaf=10,
        criterion='entropy'
    )
    dt.fit(X_train_final, y_train_encoded)
    
    pred_val = dt.predict(X_val_final)
    pred_train = dt.predict(X_train_final)
    
    acc_val = np.mean(y_val_encoded.values == pred_val)
    acc_train = np.mean(y_train_encoded.values == pred_train)
    
    info = dt.get_tree_info()
    overfitting = acc_train - acc_val
    
    print(f"Depth={depth:2d}: Train={acc_train:.4f}, Val={acc_val:.4f}, "
          f"Overfit={overfitting:+.4f}, Leaves={info['n_leaves']:3d}, "
          f"Features={info['n_features_used']:2d}/{info['n_features']:2d}")
    
    # Show which features are used for this depth
    used_feature_names = [feature_names[idx] for idx in info['used_features']]
    print(f"          Features: {', '.join(used_feature_names[:5])}", end='')
    if len(used_feature_names) > 5:
        print(f", ... (+{len(used_feature_names)-5} more)")
    else:
        print()
    
    depth_results.append({
        'depth': depth,
        'acc_val': acc_val,
        'acc_train': acc_train,
        'overfitting': overfitting,
        'leaves': info['n_leaves'],
        'n_features_used': info['n_features_used'],
        'used_features': info['used_features'],
        'model': dt
    })

# Find best depth based on validation accuracy
best_depth_result = max(depth_results, key=lambda x: x['acc_val'])
best_depth = best_depth_result['depth']

print("\n" + "-"*100)
print(f"[OK] BEST DEPTH FOUND: {best_depth}")
print(f"  Validation Accuracy: {best_depth_result['acc_val']:.4f}")
print(f"  Training Accuracy: {best_depth_result['acc_train']:.4f}")
print(f"  Overfitting Gap: {best_depth_result['overfitting']:+.4f}")
print(f"  Number of Leaves: {best_depth_result['leaves']}")
print(f"  Features Used: {best_depth_result['n_features_used']}/{len(feature_names)}")

print(f"\n[Stats] Features used in best tree (sorted by importance):")
# Get feature importances
best_model = best_depth_result['model']
used_feature_indices = best_depth_result['used_features']
feature_importances = []
for idx in used_feature_indices:
    imp = best_model.feature_importances_[idx]
    name = feature_names[idx]
    feature_importances.append((name, imp))

# Sort by importance
feature_importances.sort(key=lambda x: x[1], reverse=True)

# Print top features
print(f"  Top 15 most important features:")
for i, (name, imp) in enumerate(feature_importances[:15], 1):
    print(f"    {i:2d}. {name:45s} = {imp:.4f}")

if len(feature_importances) > 15:
    print(f"  ... and {len(feature_importances) - 15} more features")

print("="*100)

# Show detailed features breakdown for all depths


for result in depth_results:
    depth = result['depth']
    model = result['model']
    used_indices = result['used_features']
    
    # Get features with importance
    features_with_imp = []
    for idx in used_indices:
        imp = model.feature_importances_[idx]
        name = feature_names[idx]
        features_with_imp.append((name, imp))
    
    # Sort by importance
    features_with_imp.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n{'-'*100}")
    print(f"Depth={depth:2d} | Val Acc={result['acc_val']:.4f} | Features Used: {len(used_indices)}/{len(feature_names)} "
          f"| Unused: {len(feature_names) - len(used_indices)}")
    print(f"{'-'*100}")
    
    # Print top 10 features for this depth
    print(f"  Top 10 most important features (sorted by importance):")
    for i, (name, imp) in enumerate(features_with_imp[:10], 1):
        print(f"    {i:2d}. {name:45s} = {imp:.4f}")
    
    if len(features_with_imp) > 10:
        print(f"  ... and {len(features_with_imp) - 10} more features")

print("\n" + "="*100)

print(f"\n-> Best depth found: {best_depth} with Val Acc: {best_depth_result['acc_val']:.4f}")
print(f"  Overfitting: {best_depth_result['overfitting']:+.4f}")
print(f"  Leaves: {best_depth_result['leaves']}")

# Use best model from depth exploration
best_model = best_depth_result['model']
best_params = {
    'max_depth': best_depth,
    'min_samples_split': 20,
    'min_samples_leaf': 10,
    'criterion': 'entropy'
}

print("\n" + "="*100)
print(f"SELECTED MODEL: max_depth={best_depth}")
print("="*100)


QUICK DEPTH EXPLORATION - Find Best max_depth
Strategy: Test different max_depth values quickly
Fixed params: min_samples_split=20, min_samples_leaf=10, criterion=entropy

Note:
  - Total features available: 44 (original + engineered)
  - Depth = max number of questions in a decision path
  - Features used = how many different attributes tree actually needs
  - Example: Depth=5 might only use 19/44 features (other 25 not important)
----------------------------------------------------------------------------------------------------
Depth= 5: Train=0.7722, Val=0.7581, Overfit=+0.0141, Leaves= 29, Features=19/44
          Features: Course, Previous qualification (grade), Mother's qualification, Admission grade, Displaced, ... (+14 more)
Depth= 8: Train=0.8053, Val=0.7403, Overfit=+0.0650, Leaves= 79, Features=27/44
          Features: Application order, Course, Previous qualification (grade), Mother's qualification, Father's qualification, ... (+22 more)
Depth=10: Train=0.8235, Val=0.730

### Step 2: Fine-tune min_samples params with best depth
Quick test of min_samples_split and min_samples_leaf

In [177]:
print("\n" + "="*100)
print(f"FINE-TUNING min_samples with best_depth={best_depth}")
print("="*100)

# Test different min_samples combinations dengan best depth
print("Testing min_samples_split and min_samples_leaf combinations...")
print("-"*100)

finetune_configs = [
    {'split': 15, 'leaf': 5},
    {'split': 15, 'leaf': 7},
    {'split': 20, 'leaf': 5},
    {'split': 20, 'leaf': 7},
    {'split': 20, 'leaf': 10},
    {'split': 25, 'leaf': 10},
]

finetune_results = []
for config in finetune_configs:
    dt = DecisionTreeClassifier(
        max_depth=best_depth,
        min_samples_split=config['split'],
        min_samples_leaf=config['leaf'],
        criterion='entropy'
    )
    dt.fit(X_train_final, y_train_encoded)
    
    pred_val = dt.predict(X_val_final)
    pred_train = dt.predict(X_train_final)
    
    acc_val = np.mean(y_val_encoded.values == pred_val)
    acc_train = np.mean(y_train_encoded.values == pred_train)
    
    info = dt.get_tree_info()
    overfitting = acc_train - acc_val
    
    print(f"split={config['split']:2d}, leaf={config['leaf']:2d}: "
          f"Train={acc_train:.4f}, Val={acc_val:.4f}, Overfit={overfitting:+.4f}")
    
    finetune_results.append({
        'config': config,
        'acc_val': acc_val,
        'acc_train': acc_train,
        'overfitting': overfitting,
        'model': dt
    })

# Select best fine-tuned model
best_finetune = max(finetune_results, key=lambda x: x['acc_val'])

if best_finetune['acc_val'] > best_depth_result['acc_val']:
    print(f"\n-> Fine-tuning improved Val Acc: {best_depth_result['acc_val']:.4f} -> {best_finetune['acc_val']:.4f}")
    best_model = best_finetune['model']
    best_params = {
        'max_depth': best_depth,
        'min_samples_split': best_finetune['config']['split'],
        'min_samples_leaf': best_finetune['config']['leaf'],
        'criterion': 'entropy'
    }
else:
    print(f"\n-> No improvement from fine-tuning, keeping original params")

print(f"\nFinal Best Params:")
for key, val in best_params.items():
    print(f"  {key}: {val}")
print("="*100)


FINE-TUNING min_samples with best_depth=5
Testing min_samples_split and min_samples_leaf combinations...
----------------------------------------------------------------------------------------------------
split=15, leaf= 5: Train=0.7754, Val=0.7597, Overfit=+0.0158
split=15, leaf= 7: Train=0.7730, Val=0.7597, Overfit=+0.0133
split=20, leaf= 5: Train=0.7742, Val=0.7581, Overfit=+0.0162
split=20, leaf= 7: Train=0.7726, Val=0.7597, Overfit=+0.0129
split=20, leaf=10: Train=0.7722, Val=0.7581, Overfit=+0.0141
split=25, leaf=10: Train=0.7710, Val=0.7565, Overfit=+0.0146

-> Fine-tuning improved Val Acc: 0.7581 -> 0.7597

Final Best Params:
  max_depth: 5
  min_samples_split: 15
  min_samples_leaf: 5
  criterion: entropy


### Final Model: Use Best Configuration

In [178]:
print("\n" + "="*100)
print("FINAL MODEL")
print("="*100)

# Use best model found
dt_model = best_model

# Predictions
dt_pred_train = dt_model.predict(X_train_final)
dt_pred_val = dt_model.predict(X_val_final)

# Calculate metrics
dt_acc_train, dt_prec_train, dt_rec_train, dt_f1_train = calculate_metrics(
    y_train_encoded.values, dt_pred_train, "Final Model - Training Set"
)
dt_acc_val, dt_prec_val, dt_rec_val, dt_f1_val = calculate_metrics(
    y_val_encoded.values, dt_pred_val, "Final Model - Validation Set"
)

# Tree info
tree_info = dt_model.get_tree_info()
print(f"\nFinal Tree Structure:")
print(f"  Depth: {tree_info['depth']}")
print(f"  Number of leaves: {tree_info['n_leaves']}")
print(f"  Criterion: {tree_info['criterion']}")

# Feature importances
print("\n" + "="*100)
print("TOP 15 MOST IMPORTANT FEATURES")
print("="*100)
dt_model.print_feature_importances(top_n=15, feature_names=feature_names)

print("\n" + "="*100)



FINAL MODEL

Final Model - Training Set:
  Accuracy: 0.7754
  Precision per class: ['0.8249', '0.5331', '0.8037']
  Recall per class: ['0.7761', '0.3798', '0.9175']
  F1-score per class: ['0.7997', '0.4436', '0.8568']

Final Model - Validation Set:
  Accuracy: 0.7597
  Precision per class: ['0.8251', '0.4706', '0.7955']
  Recall per class: ['0.7588', '0.3604', '0.9032']
  F1-score per class: ['0.7906', '0.4082', '0.8459']

Final Tree Structure:
  Depth: 5
  Number of leaves: 29
  Criterion: entropy

TOP 15 MOST IMPORTANT FEATURES
\nTop 15 Feature Importances:
  1. Approval_Rate: 0.5575
  2. Academic_Load_Risk: 0.1348
  3. Tuition fees up to date: 0.0755
  4. Academic_Performance_Score: 0.0676
  5. Curricular units 2nd sem (approved): 0.0374
  6. Curricular units 2nd sem (enrolled): 0.0202
  7. Previous qualification (grade): 0.0146
  8. Scholarship holder: 0.0127
  9. Debtor: 0.0105
  10. Curricular units 1st sem (grade): 0.0096
  11. Average_Grade: 0.0094
  12. Total_Units_Enrolled: 

In [179]:
# Visualize tree structure
print("\n" + "="*100)
print("TREE STRUCTURE VISUALIZATION")
print("="*100)

dt_model.visualize_tree_summary(feature_names=feature_names, max_depth_show=4)

print("\nInterpretation:")
print("  - L--- : Left branch (condition is TRUE, value <= threshold)")
print("  - R--- : Right branch (condition is FALSE, value > threshold)")
print("  - LEAF: Terminal node with final class prediction")
print("  - (n=X): Number of samples reaching that node")


TREE STRUCTURE VISUALIZATION
\n====================================================================================================
TREE STRUCTURE VISUALIZATION
Tree Statistics:
  - Actual Depth: 5
  - Number of Leaves: 29
  - Features Used: 19/44
\nTree Structure (showing depth 0-4):
----------------------------------------------------------------------------------------------------
L--- Approval_Rate <= 0.80?
|   L--- Academic_Load_Risk <= 5.50?
|   |   L--- Tuition fees up to date <= 0.50?
|   |   |   L--- Curricular units 2nd sem (enrolled) <= 3.50?
|   |   |   |   L--- Previous qualification (grade) <= 141.50?
|   |   |       R--- Academic_Performance_Score <= 0.64?
|   |       R--- Previous qualification (grade) <= 142.00?
|   |       |   L--- Mother's occupation <= 6.50?
|   |           R--- LEAF: Class=0 (n=6)
|       R--- Curricular units 2nd sem (approved) <= 0.50?
|       |   L--- Curricular units 2nd sem (evaluations) <= 4.50?
|       |   |   L--- LEAF: Class=0 (n=123)
|  

In [180]:
def plot_confusion_matrix(y_true, y_pred, model_name, class_names=['Dropout', 'Enrolled', 'Graduate']):
    """
    Plot confusion matrix untuk visualisasi performance per class.
    
    Confusion Matrix menunjukkan:
    - Diagonal: prediksi benar untuk setiap class
    - Off-diagonal: misclassifications
    """
    # Create confusion matrix
    n_classes = len(class_names)
    cm = np.zeros((n_classes, n_classes), dtype=int)
    
    for true_label, pred_label in zip(y_true, y_pred):
        cm[true_label, pred_label] += 1
    
    # Print text representation
    print(f"\nConfusion Matrix - {model_name}:")
    print("\nActual \\ Predicted", end="")
    for i in range(n_classes):
        print(f"  {class_names[i]:>10}", end="")
    print()
    print("-" * (18 + 12 * n_classes))
    
    for i in range(n_classes):
        print(f"{class_names[i]:>15}", end="")
        for j in range(n_classes):
            print(f"  {cm[i, j]:>10}", end="")
        print()
    
    # Calculate per-class metrics
    print("\nPer-class Analysis:")
    for i, class_name in enumerate(class_names):
        total = np.sum(cm[i, :])
        correct = cm[i, i]
        accuracy = correct / total if total > 0 else 0
        
        # Misclassifications
        misclassified = [(class_names[j], cm[i, j]) for j in range(n_classes) if j != i and cm[i, j] > 0]
        
        print(f"\n  {class_name}:")
        print(f"    Correct: {correct}/{total} ({accuracy:.2%})")
        if misclassified:
            print(f"    Misclassified as:")
            for wrong_class, count in misclassified:
                print(f"      - {wrong_class}: {count} ({count/total:.2%})")
    
    return cm

def hyperparameter_tuning_dt(X_train, y_train, X_val, y_val):
    """
    Hyperparameter tuning untuk Decision Tree dengan grid search.
    
    Test different combinations:
    - max_depth: kedalaman tree (5, 10, 15, 20)
    - min_samples_split: minimum samples untuk split (10, 20, 30)
    - min_samples_leaf: minimum samples di leaf (5, 10, 15)
    - criterion: splitting criterion ('gini', 'entropy')
    """
    print("\n" + "="*100)
    print("HYPERPARAMETER TUNING - DECISION TREE")
    print("="*100)
    
    # Define parameter grid
    param_grid = {
        'max_depth': [5, 10, 15, 20],
        'min_samples_split': [10, 20, 30],
        'min_samples_leaf': [5, 10, 15],
        'criterion': ['gini', 'entropy']
    }
    
    best_accuracy = 0
    best_params = {}
    results = []
    
    print(f"\nParameter grid:")
    for param, values in param_grid.items():
        print(f"  {param}: {values}")
    
    print(f"\nTesting selected combinations...\n")
    
    # Test subset of combinations untuk faster execution
    test_configs = [
        {'max_depth': 10, 'min_samples_split': 20, 'min_samples_leaf': 5, 'criterion': 'gini'},
        {'max_depth': 10, 'min_samples_split': 20, 'min_samples_leaf': 5, 'criterion': 'entropy'},
        {'max_depth': 15, 'min_samples_split': 20, 'min_samples_leaf': 5, 'criterion': 'gini'},
        {'max_depth': 15, 'min_samples_split': 20, 'min_samples_leaf': 5, 'criterion': 'entropy'},
        {'max_depth': 20, 'min_samples_split': 20, 'min_samples_leaf': 5, 'criterion': 'gini'},
        {'max_depth': 20, 'min_samples_split': 20, 'min_samples_leaf': 5, 'criterion': 'entropy'},
        {'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 5, 'criterion': 'entropy'},
        {'max_depth': 15, 'min_samples_split': 30, 'min_samples_leaf': 5, 'criterion': 'entropy'},
        {'max_depth': 15, 'min_samples_split': 20, 'min_samples_leaf': 10, 'criterion': 'entropy'},
    ]
    
    for idx, config in enumerate(test_configs):
        # Train model
        dt = DecisionTreeClassifier(**config)
        dt.fit(X_train, y_train)
        
        # Evaluate
        y_pred = dt.predict(X_val)
        accuracy = np.mean(y_val == y_pred)
        
        results.append({
            **config,
            'accuracy': accuracy
        })
        
        print(f"Test {idx+1}/{len(test_configs)}: ", end="")
        print(f"depth={config['max_depth']}, split={config['min_samples_split']}, ", end="")
        print(f"leaf={config['min_samples_leaf']}, criterion={config['criterion']}")
        print(f"  -> Accuracy: {accuracy:.4f}")
        
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_params = config
    
    print("\n" + "="*100)
    print("BEST PARAMETERS FOUND:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")
    print(f"\n  Best Validation Accuracy: {best_accuracy:.4f}")
    print("="*100)
    
    return best_params, best_accuracy, results

def stratified_kfold_cv(X, y, k=5, max_depth=15, min_samples_split=20, 
                        min_samples_leaf=5, criterion='entropy'):
    """
    Stratified K-Fold Cross Validation untuk Decision Tree.
    
    Stratified memastikan proporsi class tetap sama di setiap fold.
    Ini penting untuk imbalanced dataset.
    
    Parameters:
    -----------
    X : array
        Feature matrix
    y : array  
        Target labels
    k : int
        Number of folds (default: 5)
    
    Returns:
    --------
    scores : list
        Accuracy scores untuk setiap fold
    mean_score : float
        Mean accuracy across folds
    std_score : float
        Standard deviation of accuracy
    """
    if isinstance(X, pd.DataFrame):
        X = X.values
    if isinstance(y, pd.Series):
        y = y.values
    
    print("\n" + "="*100)
    print(f"STRATIFIED {k}-FOLD CROSS VALIDATION")
    print("="*100)
    print(f"\nParameters:")
    print(f"  max_depth: {max_depth}")
    print(f"  min_samples_split: {min_samples_split}")
    print(f"  min_samples_leaf: {min_samples_leaf}")
    print(f"  criterion: {criterion}")
    print(f"\nNumber of folds: {k}")
    print(f"Total samples: {len(y)}")
    print(f"Samples per fold (approx): {len(y)//k}\n")
    
    # Use StratifiedKFold
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    
    scores = []
    fold_num = 1
    
    for train_idx, val_idx in skf.split(X, y):
        X_train_fold = X[train_idx]
        y_train_fold = y[train_idx]
        X_val_fold = X[val_idx]
        y_val_fold = y[val_idx]
        
        # Train model
        dt = DecisionTreeClassifier(
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            criterion=criterion
        )
        dt.fit(X_train_fold, y_train_fold)
        
        # Evaluate
        y_pred = dt.predict(X_val_fold)
        accuracy = np.mean(y_val_fold == y_pred)
        scores.append(accuracy)
        
        print(f"Fold {fold_num}: {accuracy:.4f} (train: {len(y_train_fold)}, val: {len(y_val_fold)})")
        fold_num += 1
    
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    
    print("\n" + "-"*100)
    print(f"Cross-Validation Results:")
    print(f"  Mean Accuracy: {mean_score:.4f}")
    print(f"  Std Deviation: {std_score:.4f}")
    print(f"  Min Accuracy: {min(scores):.4f}")
    print(f"  Max Accuracy: {max(scores):.4f}")
    print(f"  95% Confidence Interval: [{mean_score - 1.96*std_score:.4f}, {mean_score + 1.96*std_score:.4f}]")
    print("="*100)
    
    return scores, mean_score, std_score

print("="*100)
print("IMPROVEMENT FUNCTIONS LOADED")
print("="*100)
print("Available functions:")
print("1. plot_confusion_matrix() - Visualize confusion matrix with analysis")
print("2. hyperparameter_tuning_dt() - Grid search untuk Decision Tree")
print("3. stratified_kfold_cv() - K-Fold cross validation with stratification")
print("="*100)

IMPROVEMENT FUNCTIONS LOADED
Available functions:
1. plot_confusion_matrix() - Visualize confusion matrix with analysis
2. hyperparameter_tuning_dt() - Grid search untuk Decision Tree
3. stratified_kfold_cv() - K-Fold cross validation with stratification


In [181]:
def calculate_metrics(y_true, y_pred, model_name):
    """Hitung accuracy, precision, recall, F1-score untuk setiap kelas"""
    from collections import Counter
    
    # Accuracy
    accuracy = np.mean(y_true == y_pred)
    
    # Per-class metrics
    classes = np.unique(y_true)
    precision_per_class = []
    recall_per_class = []
    f1_per_class = []
    
    for cls in classes:
        # True Positives, False Positives, False Negatives
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))
        
        # Precision, Recall, F1
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        precision_per_class.append(precision)
        recall_per_class.append(recall)
        f1_per_class.append(f1)
    
    print(f"\n{model_name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision per class: {[f'{p:.4f}' for p in precision_per_class]}")
    print(f"  Recall per class: {[f'{r:.4f}' for r in recall_per_class]}")
    print(f"  F1-score per class: {[f'{f:.4f}' for f in f1_per_class]}")
    
    return accuracy, precision_per_class, recall_per_class, f1_per_class

## D. Submission
To predict the test set target feature and submit the results, do the following:
1. Preprocess test data dengan pipeline yang sama seperti training data
2. Apply transformations: outlier clipping, feature engineering, scaling, encoding, PCA
3. Predict menggunakan trained Decision Tree model
4. Generate submission file dengan format: Student_ID, Target

In [182]:
# Optional: Run confusion matrix visualization
cm_dt = plot_confusion_matrix(y_val_encoded.values, dt_pred_val, "Decision Tree - Validation Set")


Confusion Matrix - Decision Tree - Validation Set:

Actual \ Predicted     Dropout    Enrolled    Graduate
------------------------------------------------------
        Dropout         151          23          25
       Enrolled          24          40          47
       Graduate           8          22         280

Per-class Analysis:

  Dropout:
    Correct: 151/199 (75.88%)
    Misclassified as:
      - Enrolled: 23 (11.56%)
      - Graduate: 25 (12.56%)

  Enrolled:
    Correct: 40/111 (36.04%)
    Misclassified as:
      - Dropout: 24 (21.62%)
      - Graduate: 47 (42.34%)

  Graduate:
    Correct: 280/310 (90.32%)
    Misclassified as:
      - Dropout: 8 (2.58%)
      - Enrolled: 22 (7.10%)


In [183]:
# Optional: Run hyperparameter tuning
# Uncomment untuk mencari best hyperparameters
# best_params, best_acc, tuning_results = hyperparameter_tuning_dt(
#     X_train_final, y_train_encoded.values, 
#     X_val_final, y_val_encoded.values
# )

In [184]:
# Optional: Run cross-validation untuk evaluate model stability
# Uncomment untuk run cross-validation
# cv_scores, cv_mean, cv_std = stratified_kfold_cv(
#     X_train_final, y_train_encoded.values,
#     k=5,
#     max_depth=15,
#     min_samples_split=20,
#     min_samples_leaf=5,
#     criterion='entropy'
# )

In [185]:
test_df = pd.read_csv("../data/test.csv")
test_ids = test_df["Student_ID"]
X_test = test_df.drop(columns=["Student_ID"])

print("="*100)
print("PREPROCESSING TEST DATA")
print("="*100)

# 1. Handle numerical outliers
print("\n1. Clipping numerical outliers...")
outlier_count = 0
for col in continuous_cols:
    if col in outlier_info and col in X_test.columns:
        lower = outlier_info[col]['lower_bound']
        upper = outlier_info[col]['upper_bound']
        before_clip = ((X_test[col] < lower) | (X_test[col] > upper)).sum()
        X_test[col] = X_test[col].clip(lower, upper)
        outlier_count += before_clip
print(f"   {outlier_count} outliers clipped")

# 2. Feature engineering
print("\n2. Feature engineering...")
X_test['Total_Units_Approved'] = (
    X_test['Curricular units 1st sem (approved)'] + 
    X_test['Curricular units 2nd sem (approved)']
)
X_test['Total_Units_Enrolled'] = (
    X_test['Curricular units 1st sem (enrolled)'] + 
    X_test['Curricular units 2nd sem (enrolled)']
)
X_test['Approval_Rate'] = np.where(
    X_test['Total_Units_Enrolled'] > 0,
    X_test['Total_Units_Approved'] / X_test['Total_Units_Enrolled'],
    0
)
X_test['Average_Grade'] = (
    X_test['Curricular units 1st sem (grade)'] + 
    X_test['Curricular units 2nd sem (grade)']
) / 2
X_test['Grade_Difference'] = (
    X_test['Curricular units 2nd sem (grade)'] - 
    X_test['Curricular units 1st sem (grade)']
)

# Advanced interaction features (same as training)
X_test['Academic_Performance_Score'] = (
    0.6 * X_test['Approval_Rate'] + 
    0.4 * (X_test['Average_Grade'] / X_test['Average_Grade'].max())
)
X_test['Grade_Trend_Binary'] = (X_test['Grade_Difference'] > 0).astype(int)
X_test['Academic_Load_Risk'] = X_test['Total_Units_Enrolled'] - X_test['Total_Units_Approved']

print("   8 features created (5 basic + 3 interaction)")

# 3. Skip scaling (Decision Tree doesn't need it)
print("\n3. Scaling skipped (Decision Tree is scale-invariant)")
X_test_scaled = X_test.copy()

# 4. Label encoding
print("\n4. Label encoding...")
X_test_encoded = label_encoder.transform(X_test_scaled, categorical_cols)
print(f"   {len(categorical_cols)} categorical columns encoded")

# 5. Skip PCA (using all features for Decision Tree)
print("\n5. PCA skipped (using all features)")
X_test_final = X_test_encoded.values
print(f"   Total features: {X_test_final.shape[1]}")

print("\n" + "="*100)
print("GENERATING PREDICTIONS")
print("="*100)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Decision Tree prediction
print("\n" + "="*100)
print("GENERATING PREDICTIONS")
print("="*100)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Decision Tree prediction
print("\nDecision Tree (CART) - Generating submission...")
dt_pred_encoded = dt_model.predict(X_test_final)
dt_pred = pd.Series([target_encoder.inverse_mapping_['Target'][pred] for pred in dt_pred_encoded])

submission_dt = pd.DataFrame({
    "Student_ID": test_ids,
    "Target": dt_pred
})
dt_path = f"../submission/submission_{timestamp}_DTL.csv"
submission_dt.to_csv(dt_path, index=False)

print("\n" + "="*100)
print("SUBMISSION FILE GENERATED")
print("="*100)
print(f"Model: Decision Tree (CART)")
print(f"File: {dt_path}")
print(f"Total predictions: {len(test_ids)}")
print(f"Validation Accuracy: {dt_acc_val:.4f}")
print(f"Training Accuracy: {dt_acc_train:.4f}")
print(f"Tree Depth: {tree_info['depth']}")
print(f"Number of Leaves: {tree_info['n_leaves']}")
print("="*100)


PREPROCESSING TEST DATA

1. Clipping numerical outliers...
   11 outliers clipped

2. Feature engineering...
   8 features created (5 basic + 3 interaction)

3. Scaling skipped (Decision Tree is scale-invariant)

4. Label encoding...
   18 categorical columns encoded

5. PCA skipped (using all features)
   Total features: 44

GENERATING PREDICTIONS

GENERATING PREDICTIONS

Decision Tree (CART) - Generating submission...

SUBMISSION FILE GENERATED
Model: Decision Tree (CART)
File: ../submission/submission_20251201_185658_DTL.csv
Total predictions: 1328
Validation Accuracy: 0.7597
Training Accuracy: 0.7754
Tree Depth: 5
Number of Leaves: 29


# 5. Error Analysis

Based on all the process you have done until the modeling and evaluation step, write an analysis to support each steps you have taken to solve this problem. Write the analysis using the markdown block. Some questions that may help you in writing the analysis:

- Does my model perform better in predicting one class than the other? If so, why is that?
- What hyperparameters work best for the Decision Tree and why?
- Is it better for me to impute or drop the missing data? Why?
- Does feature scaling help improve my model performance?
- How does the criterion (gini vs entropy) affect the model?
- What does the confusion matrix tell us about the model's behavior?
- etc...

`Provide your analysis here`